**Exploración de Datos**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_eda = pd.read_csv(ruta_dataset)

col_centro = 'id_centro'

print(f"Iniciando Análisis Exploratorio Global por {col_centro}...")
print(f"Centros detectados: {df_eda[col_centro].nunique()}")

sns.set_theme(style="whitegrid")

paleta_centros = {
    110557: '#d62728',  # Rojo
    120069: '#5fa2ce',  # Azul claro
    110745: '#2ca02c'   # Verde
}

# ==========================================
# 2. Análisis del Ciclo Biológico (Boxplots)
# ==========================================
fig_bio, axes_bio = plt.subplots(3, 1, figsize=(14, 12), dpi=150)

estadios = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
titulos_bio = ['Carga de Juveniles', 'Carga de Adultos Móviles', 'Carga de Hembras Ovígeras (Umbral Crítico)']

for i, estadio in enumerate(estadios):
    sns.boxplot(x=col_centro, y=estadio, data=df_eda, ax=axes_bio[i], 
                hue=col_centro, palette=paleta_centros, legend=False, 
                width=0.6, fliersize=3)
    
    axes_bio[i].set_title(titulos_bio[i], fontweight='bold')
    axes_bio[i].set_xlabel('')
    axes_bio[i].set_ylabel('Cantidad')
    axes_bio[i].tick_params(axis='x', rotation=45)
    
    axes_bio[i].set_ylim(bottom=0)
    
    if estadio == 'Hembras_Ovigeras':
        axes_bio[i].axhline(y=3, color='black', linestyle='--', alpha=0.7, label='Umbral 3 Hembras')
        axes_bio[i].legend(loc='upper right')

fig_bio.suptitle('Radiografía Epidemiológica: Ciclo de Vida del Caligus por Centro', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(carpeta_destino, "EDA_Biologia_Completa.png"), bbox_inches='tight')
plt.show()

# ==========================================
# 3. Análisis Termodinámico y Físico (Violin Plots)
# ==========================================
fig_fis, axes_fis = plt.subplots(4, 1, figsize=(14, 16), dpi=150)

vars_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m']
titulos_fisicos = ['Perfil Térmico (°C)', 'Perfil Osmótico (psu)', 'Profundidad Capa de Mezcla (m)', 'Altura del Mar (m)']

for i, var in enumerate(vars_fisicas):
    sns.violinplot(x=col_centro, y=var, data=df_eda, ax=axes_fis[i], 
                   hue=col_centro, palette=paleta_centros, legend=False, 
                   inner="quartile", cut=0) 
    
    axes_fis[i].set_title(titulos_fisicos[i], fontweight='bold')
    axes_fis[i].set_xlabel('')
    axes_fis[i].set_ylabel(var)
    axes_fis[i].tick_params(axis='x', rotation=45)
    
   
    if df_eda[var].min() >= 0:
        axes_fis[i].set_ylim(bottom=0)

fig_fis.suptitle('Radiografía Oceanográfica: Estructura de la Columna de Agua por Centro', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(carpeta_destino, "EDA_Fisica_Completa.png"), bbox_inches='tight')
plt.show()

# ==========================================
# 4. Tabla de Resumen Estadístico Maestro
# ==========================================
print("\n--- MATRIZ DE RIESGO Y CONDICIONES POR CENTRO ---")
resumen_centros = df_eda.groupby(col_centro).agg(
    Semanas=(col_centro, 'count'),
    Temp_Media=('Temperatura_C', 'mean'),
    Sal_Media=('Salinidad_psu', 'mean'),
    Capa_Mezcla_Media=('Capa_Mezcla_m', 'mean'),
    Hembras_Media=('Hembras_Ovigeras', 'mean'),
    Hembras_Max=('Hembras_Ovigeras', 'max'),
    Juv_Max=('Juveniles', 'max')
).round(2).sort_values(by='Hembras_Media', ascending=False)

print(resumen_centros)

ruta_tabla = os.path.join(carpeta_destino, "Resumen_Centros.csv")
resumen_centros.to_csv(ruta_tabla)
print(f"\nTabla exportada exitosamente a: {ruta_tabla}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_ts = pd.read_csv(ruta_dataset)


df_ts['Fecha'] = pd.to_datetime(df_ts['Fecha'])

col_centro = 'id_centro'
print("Generando Series de Tiempo Longitudinales...")

sns.set_theme(style="whitegrid")

# Diccionario de colores exactos
paleta_centros = {
    110557: '#d62728',  # Rojo
    120069: '#5fa2ce',  # Azul claro
    110745: '#2ca02c'   # Verde
}


años_unicos = df_ts['Año'].unique()
fechas_inicio_año = [pd.to_datetime(f'{year}-01-01') for year in años_unicos]

# ==========================================
# 2. Series de Tiempo: Ciclo Biológico
# ==========================================
fig_bio, axes_bio = plt.subplots(3, 1, figsize=(15, 12), dpi=150, sharex=True)

estadios = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
titulos_bio = ['Evolución Temporal: Juveniles', 'Evolución Temporal: Adultos Móviles', 'Evolución Temporal: Hembras Ovígeras']

for i, estadio in enumerate(estadios):
    sns.lineplot(x='Fecha', y=estadio, hue=col_centro, data=df_ts, 
                 palette=paleta_centros, ax=axes_bio[i], linewidth=1.5, alpha=0.8)
    
    axes_bio[i].set_title(titulos_bio[i], fontweight='bold')
    axes_bio[i].set_ylabel('Promedio por Pez')
    
    
    axes_bio[i].set_ylim(bottom=0)
    
   
    for inicio_año in fechas_inicio_año:
        axes_bio[i].axvline(x=inicio_año, color='gray', linestyle=':', alpha=0.7)
    
    
    if estadio == 'Hembras_Ovigeras':
        axes_bio[i].axhline(y=3, color='black', linestyle='--', linewidth=2, label='Umbral Crítico (3 Hembras)')
        axes_bio[i].legend(loc='upper left')
    else:
        axes_bio[i].get_legend().remove()

axes_bio[-1].set_xlabel('Fecha')
fig_bio.suptitle('Dinámica Epidemiológica Temporal del Caligus rogercresseyi', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(carpeta_destino, "TS_Biologia_Temporal.png"), bbox_inches='tight')
plt.show()

# ==========================================
# 3. Series de Tiempo: Oceanografía y Física
# ==========================================
fig_fis, axes_fis = plt.subplots(4, 1, figsize=(15, 16), dpi=150, sharex=True)

vars_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m']
titulos_fisicos = ['Fluctuación Térmica (°C)', 'Fluctuación de Salinidad (psu)', 'Profundidad de la Capa de Mezcla (m)', 'Altura del Mar (m)']

for i, var in enumerate(vars_fisicas):
    sns.lineplot(x='Fecha', y=var, hue=col_centro, data=df_ts, 
                 palette=paleta_centros, ax=axes_fis[i], linewidth=1.2, alpha=0.8)
    
    axes_fis[i].set_title(titulos_fisicos[i], fontweight='bold')
    axes_fis[i].set_ylabel(var)
    

    if df_ts[var].min() >= 0:
        axes_fis[i].set_ylim(bottom=0)
        

    if var == 'Capa_Mezcla_m':
        axes_fis[i].invert_yaxis()
    

    for inicio_año in fechas_inicio_año:
        axes_fis[i].axvline(x=inicio_año, color='gray', linestyle=':', alpha=0.7)
        
    if i == 0:
        axes_fis[i].legend(title='ID Centro', loc='upper right')
    else:
        axes_fis[i].get_legend().remove()

axes_fis[-1].set_xlabel('Fecha')
fig_fis.suptitle('Dinámica Oceanográfica y Termodinámica del Fiordo', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(carpeta_destino, "TS_Fisica_Temporal.png"), bbox_inches='tight')
plt.show()

print(f"\nRenderizado completo. Gráficos temporales exportados en: {carpeta_destino}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_est = pd.read_csv(ruta_dataset)
df_est['Fecha'] = pd.to_datetime(df_est['Fecha'])

col_centro = 'id_centro'
print("Iniciando Análisis Estacional...")

# ==========================================
# 2. Ingeniería de Características: Estaciones del Año
# ==========================================

df_est['Mes'] = df_est['Fecha'].dt.month


def asignar_estacion(mes):
    if mes in [12, 1, 2]:
        return 'Verano'
    elif mes in [3, 4, 5]:
        return 'Otoño'
    elif mes in [6, 7, 8]:
        return 'Invierno'
    elif mes in [9, 10, 11]:
        return 'Primavera'

df_est['Estacion'] = df_est['Mes'].apply(asignar_estacion)

# Ordenamos categóricamente las estaciones para que los gráficos tengan lógica cronológica
orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
df_est['Estacion'] = pd.Categorical(df_est['Estacion'], categories=orden_estaciones, ordered=True)

# Paleta de colores para las estaciones (cálidos a fríos)
paleta_estaciones = {
    'Verano': '#d62728',     # Rojo
    'Otoño': '#ff7f0e',      # Naranja
    'Invierno': '#1f77b4',   # Azul
    'Primavera': '#2ca02c'   # Verde
}

# ==========================================
# 3. Análisis Visual Estacional por Centro
# ==========================================
sns.set_theme(style="whitegrid")

# Creamos un panel con 3 variables clave para entender la estación
fig, axes = plt.subplots(3, 1, figsize=(15, 14), dpi=150)

# A. Validación Térmica (Para asegurar que las estaciones tienen sentido físico)
sns.boxplot(x=col_centro, y='Temperatura_C', hue='Estacion', data=df_est, 
            ax=axes[0], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[0].set_title('Perfil Térmico Estacional por Centro', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Temperatura (°C)')
axes[0].legend(title='Estación', loc='upper right')

# B. Dinámica de la Columna de Agua (Capa de Mezcla)
sns.boxplot(x=col_centro, y='Capa_Mezcla_m', hue='Estacion', data=df_est, 
            ax=axes[1], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[1].set_title('Profundidad de la Capa de Mezcla Estacional', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Profundidad (m)')
axes[1].invert_yaxis() # Invertimos para ver la profundidad hacia abajo
axes[1].get_legend().remove()

# C. Carga Epidemiológica (Hembras Ovígeras)
sns.boxplot(x=col_centro, y='Hembras_Ovigeras', hue='Estacion', data=df_est, 
            ax=axes[2], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[2].set_title('Impacto Epidemiológico Estacional (Hembras Ovígeras)', fontweight='bold')
axes[2].set_xlabel('Centro de Cultivo')
axes[2].set_ylabel('Cantidad Promedio')
axes[2].set_ylim(bottom=0)
axes[2].axhline(y=3, color='black', linestyle='--', alpha=0.7, label='Umbral 3 Hembras')
axes[2].get_legend().remove()

fig.suptitle('Comparativa Estacional Intra e Inter-Centros', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(carpeta_destino, "EDA_Estacional_Centros.png"), bbox_inches='tight')
plt.show()

# ==========================================
# 4. Matriz Cuantitativa Estacional
# ==========================================
print("\n--- PROMEDIO DE HEMBRAS OVÍGERAS POR CENTRO Y ESTACIÓN ---")
# Creamos una tabla pivote (matriz cruzada) para ver los promedios numéricos exactos
matriz_estacional = pd.pivot_table(
    df_est, 
    values='Hembras_Ovigeras', 
    index=col_centro, 
    columns='Estacion', 
    aggfunc='mean'
).round(2)

print(matriz_estacional)

# Exportar el dataset con la nueva columna 'Estacion' si deseas conservarlo para los modelos
# df_est.to_csv(ruta_dataset, index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Necesario para el cálculo de magnitud de corriente
import os

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
# (Asegúrate de que la ruta del dataset sea la correcta)
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_est = pd.read_csv(ruta_dataset)
df_est['Fecha'] = pd.to_datetime(df_est['Fecha'])

col_centro = 'id_centro'
print("Iniciando Análisis Estacional Expandido Integrado (8 Variables)...")

# ==========================================
# 2. Ingeniería de Características: Estaciones del Año y Derivadas
# ==========================================

df_est['Mes'] = df_est['Fecha'].dt.month

def asignar_estacion(mes):
    if mes in [12, 1, 2]:
        return 'Verano'
    elif mes in [3, 4, 5]:
        return 'Otoño'
    elif mes in [6, 7, 8]:
        return 'Invierno'
    elif mes in [9, 10, 11]:
        return 'Primavera'

df_est['Estacion'] = df_est['Mes'].apply(asignar_estacion)


orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
df_est['Estacion'] = pd.Categorical(df_est['Estacion'], categories=orden_estaciones, ordered=True)


paleta_estaciones = {
    'Verano': '#d62728',     # Rojo
    'Otoño': '#ff7f0e',      # Naranja
    'Invierno': '#1f77b4',   # Azul
    'Primavera': '#2ca02c'   # Verde
}


df_est['Velocidad_Corriente_m_s'] = np.sqrt(df_est['Corriente_E_m_s']**2 + df_est['Corriente_N_m_s']**2)

# ==========================================
# 3. Análisis Visual Estacional por Centro (Dashboard de 8 Paneles)
# ==========================================
sns.set_theme(style="whitegrid")


n_subplots = 8

fig, axes = plt.subplots(n_subplots, 1, figsize=(15, 5.5 * n_subplots), dpi=150) 

sns.boxplot(x=col_centro, y='Temperatura_C', hue='Estacion', data=df_est,
            ax=axes[0], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[0].set_title('Perfil Térmico Estacional por Centro', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Temperatura (°C)')
axes[0].legend(title='Estación', loc='upper right')


sns.boxplot(x=col_centro, y='Capa_Mezcla_m', hue='Estacion', data=df_est,
            ax=axes[1], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[1].set_title('Profundidad de la Capa de Mezcla Estacional', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Profundidad (m)')
axes[1].invert_yaxis() 
axes[1].get_legend().remove()


sns.boxplot(x=col_centro, y='Hembras_Ovigeras', hue='Estacion', data=df_est,
            ax=axes[2], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[2].set_title('Impacto Epidemiológico Estacional (Hembras Ovígeras)', fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_ylabel('Cantidad Promedio')
axes[2].set_ylim(bottom=0)
axes[2].axhline(y=3, color='black', linestyle='--', alpha=0.7, label='Umbral Crítico (3 Hembras)')
axes[2].get_legend().remove()




sns.boxplot(x=col_centro, y='Salinidad_psu', hue='Estacion', data=df_est,
            ax=axes[3], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[3].set_title('Perfil de Salinidad Estacional por Centro', fontweight='bold')
axes[3].set_xlabel('')
axes[3].set_ylabel('Salinidad (psu)')
axes[3].get_legend().remove()


sns.boxplot(x=col_centro, y='Velocidad_Corriente_m_s', hue='Estacion', data=df_est,
            ax=axes[4], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[4].set_title('Evolución Estacional de Velocidad de Corriente', fontweight='bold')
axes[4].set_xlabel('')
axes[4].set_ylabel('Velocidad (m/s)')
axes[4].set_ylim(bottom=0)
axes[4].get_legend().remove()


sns.boxplot(x=col_centro, y='Altura_Mar_m', hue='Estacion', data=df_est,
            ax=axes[5], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[5].set_title('Fluctuación Estacional de Altura del Mar', fontweight='bold')
axes[5].set_xlabel('')
axes[5].set_ylabel('Altura del Mar (m)')
axes[5].get_legend().remove()


sns.boxplot(x=col_centro, y='Juveniles', hue='Estacion', data=df_est,
            ax=axes[6], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[6].set_title('Carga Estacional de Juveniles por Centro', fontweight='bold')
axes[6].set_xlabel('')
axes[6].set_ylabel('Cantidad Promedio')
axes[6].set_ylim(bottom=0)
axes[6].get_legend().remove()


sns.boxplot(x=col_centro, y='Adultos_Moviles', hue='Estacion', data=df_est,
            ax=axes[7], palette=paleta_estaciones, width=0.7, fliersize=2)
axes[7].set_title('Carga Estacional de Adultos Móviles por Centro', fontweight='bold')
axes[7].set_xlabel('Centro de Cultivo')
axes[7].set_ylabel('Cantidad Promedio')
axes[7].set_ylim(bottom=0)
axes[7].get_legend().remove()


fig.suptitle('Análisis Estacional Integrado y Expandido de Variables Físicas y Biológicas', fontsize=16, fontweight='bold', y=1.002)
plt.tight_layout(rect=[0, 0, 1, 1.0])


ruta_grafico_expandido = os.path.join(carpeta_destino, "EDA_Estacional_Completa_Expandida_8Variables.png")
plt.savefig(ruta_grafico_expandido, bbox_inches='tight')
plt.show()

print(f"\nDashboard expandido e integrado generado exitosamente. Gráfico maestro exportado en: {ruta_grafico_expandido}")

In [ ]:
ruta_dataset_enriquecido = "data/Dataset_Maestro_Estacional_ML.csv"

df_est.to_csv(ruta_dataset_enriquecido, index=False)
print(f"\nDataset estacional enriquecido exportado a: {ruta_dataset_enriquecido}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_ts = pd.read_csv(ruta_dataset)
df_ts['Fecha'] = pd.to_datetime(df_ts['Fecha'])

col_centro = 'id_centro'
print("Iniciando Renderizado de Series de Tiempo con Fondos Estacionales...")

# ==========================================
# 2. Ingeniería de Características: Estaciones y Derivadas
# ==========================================
df_ts['Mes'] = df_ts['Fecha'].dt.month

def asignar_estacion(mes):
    if mes in [12, 1, 2]: return 'Verano'
    elif mes in [3, 4, 5]: return 'Otoño'
    elif mes in [6, 7, 8]: return 'Invierno'
    elif mes in [9, 10, 11]: return 'Primavera'

df_ts['Estacion'] = df_ts['Mes'].apply(asignar_estacion)
df_ts['Velocidad_Corriente_m_s'] = np.sqrt(df_ts['Corriente_E_m_s']**2 + df_ts['Corriente_N_m_s']**2)

# ==========================================
# 3. Configuración Visual (Paletas y Fondos)
# ==========================================
sns.set_theme(style="white")


paleta_centros = {
    110557: '#d62728',  # Rojo
    120069: '#5fa2ce',  # Azul claro
    110745: '#2ca02c'   # Verde
}


color_fondo_estacion = {
    'Verano': "#EE291B",    
    'Otoño': "#DBB433",    
    'Invierno': "#3F92EB", 
    'Primavera': "#67F052" 
}

fechas_unicas = df_ts[['Fecha', 'Estacion']].drop_duplicates().sort_values('Fecha')

# ==========================================
# 4. Renderizado del Dashboard Temporal
# ==========================================
n_subplots = 8
fig, axes = plt.subplots(n_subplots, 1, figsize=(15, 5.5 * n_subplots), dpi=150, sharex=True)


variables = [
    'Temperatura_C', 'Capa_Mezcla_m', 'Hembras_Ovigeras', 'Salinidad_psu',
    'Velocidad_Corriente_m_s', 'Altura_Mar_m', 'Juveniles', 'Adultos_Moviles'
]

titulos = [
    'Perfil Térmico Longitudinal (°C)', 'Profundidad Capa de Mezcla (m)', 'Impacto Epidemiológico: Hembras Ovígeras',
    'Perfil Osmótico Longitudinal (psu)', 'Magnitud de Corriente Hidrodinámica (m/s)', 'Fluctuación Altura del Mar (m)',
    'Carga Longitudinal de Juveniles', 'Carga Longitudinal de Adultos Móviles'
]

for i, var in enumerate(variables):

    sns.lineplot(x='Fecha', y=var, hue=col_centro, data=df_ts, 
                 palette=paleta_centros, ax=axes[i], linewidth=1.5, alpha=0.8)
    
    
    for estacion, color in color_fondo_estacion.items():
        axes[i].fill_between(fechas_unicas['Fecha'], 0, 1, 
                             where=(fechas_unicas['Estacion'] == estacion), 
                             color=color, alpha=0.3, transform=axes[i].get_xaxis_transform())


    axes[i].set_title(titulos[i], fontweight='bold')
    axes[i].set_ylabel(var)
    
    
    if df_ts[var].min() >= 0:
        axes[i].set_ylim(bottom=0)
        

    if var == 'Capa_Mezcla_m':
        axes[i].invert_yaxis()
        

    if var == 'Hembras_Ovigeras':
        axes[i].axhline(y=3, color='black', linestyle='--', linewidth=2, label='Umbral Crítico (3 Hembras)')
    

    if i == 0:
        axes[i].legend(title='ID Centro', loc='upper right')
    elif var == 'Hembras_Ovigeras':
        axes[i].legend(loc='upper right')
    else:
        axes[i].get_legend().remove()

axes[-1].set_xlabel('Fecha de Medición')
fig.suptitle('Análisis Longitudinal Integrado con Contexto Estacional', fontsize=18, fontweight='bold', y=1.002)

from matplotlib.patches import Patch
leyenda_estaciones = [Patch(facecolor=color_fondo_estacion[est], label=est, alpha=0.5) for est in color_fondo_estacion]
fig.legend(handles=leyenda_estaciones, loc='upper center', bbox_to_anchor=(0.5, 0.995), ncol=4, title="Estaciones del Año")

plt.tight_layout(rect=[0, 0, 1, 0.98])
ruta_grafico_ts = os.path.join(carpeta_destino, "TS_Estacional_8Variables_Fondos.png")
plt.savefig(ruta_grafico_ts, bbox_inches='tight')
plt.show()

print(f"\nDashboard temporal generado. Gráfico con fondos exportado a: {ruta_grafico_ts}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración y Mapeo de Columnas
# ==========================================
ruta_raw = "data/Cargas_parasitarias_2012_2024.xlsx"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)


col_centro_raw = 'Código Centro'
col_semana_raw = 'semana'
col_año_raw = 'Año'
col_juv_raw = 'Prom. Juveniles'
col_adultos_raw = 'Prom. Adultos Móviles'
col_hembras_raw = 'Prom. Hembras Ovígeras'

centros_objetivo = [110557, 120069, 110745]

print("Cargando datos crudos y procesando serie de tiempo...")
df_raw = pd.read_excel(ruta_raw)


df_ts = df_raw[df_raw[col_centro_raw].isin(centros_objetivo)].copy()
df_ts = df_ts[(df_ts[col_año_raw] >= 2020) & (df_ts[col_año_raw] <= 2025)].copy()

# ==========================================
# 2. Ingeniería de Fechas para Serie de Tiempo
# ==========================================

df_ts['Fecha_Continua'] = pd.to_datetime(df_ts[col_año_raw].astype(str), format='%Y') + pd.to_timedelta(df_ts[col_semana_raw] * 7, unit='days')

# Ordenamos estrictamente por fecha
df_ts = df_ts.sort_values(by=[col_centro_raw, 'Fecha_Continua'])

# ==========================================
# 3. Renderizado del Dashboard Longitudinal (Crudo)
# ==========================================
sns.set_theme(style="whitegrid")
paleta_centros = {110557: '#d62728', 120069: '#5fa2ce', 110745: '#2ca02c'}

fig, axes = plt.subplots(3, 1, figsize=(15, 14), dpi=150, sharex=True)

variables_biologicas = [col_juv_raw, col_adultos_raw, col_hembras_raw]
titulos = ['Evolución Cruda: Juveniles', 'Evolución Cruda: Adultos Móviles', 'Evolución Cruda: Hembras Ovígeras (Umbral Crítico)']

for i, var in enumerate(variables_biologicas):

    sns.lineplot(x='Fecha_Continua', y=var, hue=col_centro_raw, data=df_ts, 
                 palette=paleta_centros, ax=axes[i], 
                 marker="o", markersize=4, linewidth=1.2, alpha=0.8)
    
    axes[i].set_title(titulos[i], fontweight='bold')
    axes[i].set_ylabel('Promedio por Pez')
    axes[i].set_ylim(bottom=0)
    

    años_presentes = df_ts[col_año_raw].unique()
    for año in años_presentes:
        inicio_inv = pd.to_datetime(f"{año}-06-15")
        fin_inv = pd.to_datetime(f"{año}-08-31")
        axes[i].axvspan(inicio_inv, fin_inv, color='#e6f2ff', alpha=0.6, 
                        label='Invierno Austral' if (i==0 and año==años_presentes[0]) else "")


    if var == col_hembras_raw:
        axes[i].axhline(y=3, color='black', linestyle='--', linewidth=1.5, label='Umbral 3 Hembras')
        axes[i].legend(loc='upper right')
    elif i == 0:
        axes[i].legend(title='Centro', loc='upper right')
    else:
        axes[i].get_legend().remove()

axes[-1].set_xlabel("Fecha de Medición (2020 - 2025)")
fig.suptitle('Auditoría de Datos Crudos: Flujo del Ciclo de Vida del Caligus', fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
ruta_grafico_ts_raw = os.path.join(carpeta_destino, "Auditoria_SerieTiempo_Cruda_2020_2025.png")
plt.savefig(ruta_grafico_ts_raw, bbox_inches='tight')
plt.show()

print(f"\nSerie de tiempo generada. Gráfico exportado a: {ruta_grafico_ts_raw}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_corr = pd.read_csv(ruta_dataset)
col_centro = 'id_centro'

print("Generando Matrices Cuadradas Completas por Centro...")

# ==========================================
# 2. Ingeniería de Características Base
# ==========================================
df_corr['Velocidad_Corriente_m_s'] = np.sqrt(df_corr['Corriente_E_m_s']**2 + df_corr['Corriente_N_m_s']**2)

variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_biologicas = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
variables_matriz = variables_fisicas + variables_biologicas

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {110557: "Centro 110557 (Rojo)", 120069: "Centro 120069 (Azul)", 110745: "Centro 110745 (Verde)"}

# ==========================================
# 3. Renderizado de Paneles Cuadrados Completos
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(24, 8), dpi=150)
sns.set_theme(style="white")

cmap = sns.diverging_palette(230, 20, as_cmap=True)

for idx, centro in enumerate(centros_objetivo):

    df_centro_local = df_corr[df_corr[col_centro] == centro][variables_matriz].dropna().copy()
    

    scaler = StandardScaler()
    df_escalado_local = pd.DataFrame(scaler.fit_transform(df_centro_local), columns=variables_matriz)
    
   
    matriz_local = df_escalado_local.corr(method='spearman')
    
    
    mostrar_cbar = True if idx == 2 else False
    
    sns.heatmap(matriz_local, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f", ax=axes[idx],
                cbar=mostrar_cbar, annot_kws={"size": 10},
                cbar_kws={"shrink": .8, "label": "Coeficiente Rho de Spearman"} if mostrar_cbar else None)
    
    axes[idx].set_title(nombres_centros[centro], fontsize=14, fontweight='bold', pad=15)
    
    axes[idx].set_xticklabels(variables_matriz, rotation=45, ha='right', fontsize=10)
    if idx == 0:
        axes[idx].set_yticklabels(variables_matriz, rotation=0, fontsize=10)
    else:
        axes[idx].set_yticklabels([])

plt.suptitle('Comparativa Local de Interacción Física-Biología (Matrices Cuadradas Completas)', 
             fontsize=18, fontweight='bold', y=1.05)

plt.tight_layout()
ruta_heatmap_cuadrado = os.path.join(carpeta_destino, "Heatmap_Correlacion_Cuadrado_Centros.png")
plt.savefig(ruta_heatmap_cuadrado, bbox_inches='tight')
plt.show()

print(f"\nMatrices cuadradas completadas. Imagen exportada en: {ruta_heatmap_cuadrado}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os

# ==========================================
# 1. CONFIGURACIÓN DE ENCABEZADOS
# ==========================================
ruta_raw = "data/Cargas_parasitarias_2012_2024.xlsx"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

col_centro_raw = 'Código Centro'
col_semana_raw = 'semana'
col_año_raw = 'Año'

col_temp_insitu = 'Temperatura'
col_sal_insitu  = 'Salinidad'

col_juv_raw     = 'Prom. Juveniles'
col_adultos_raw = 'Prom. Adultos Móviles'
col_hembras_raw = 'Prom. Hembras Ovígeras'

variables_matriz = [
    col_temp_insitu, col_sal_insitu, 
    col_juv_raw, col_adultos_raw, col_hembras_raw
]

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {110557: "Centro 110557 (Rojo)", 120069: "Centro 120069 (Azul)", 110745: "Centro 110745 (Verde)"}

# ==========================================
# 2. Carga y Limpieza Local de Datos (Corregido)
# ==========================================
print("Cargando datos originales de Sernapesca para Ground-Truthing...")
df_raw = pd.read_excel(ruta_raw)


df_filtrado_centros = df_raw[df_raw[col_centro_raw].isin(centros_objetivo)].copy()

for col in variables_matriz:
    df_filtrado_centros[col] = pd.to_numeric(df_filtrado_centros[col], errors='coerce')

# ==========================================
# 3. Renderizado de Matrices Cuadradas In-Situ
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(22, 7), dpi=150)
sns.set_theme(style="white")
cmap = sns.diverging_palette(230, 20, as_cmap=True)

for idx, centro in enumerate(centros_objetivo):
    # Aislamiento y eliminación de los verdaderos nulos
    df_centro_local = df_filtrado_centros[df_filtrado_centros[col_centro_raw] == centro][variables_matriz].dropna().copy()
    
    if len(df_centro_local) == 0:
        print(f"Error: No se encontraron registros válidos para el centro {centro} después de la limpieza.")
        continue
        
    # Estandarización de las mediciones locales (Z-score)
    scaler = StandardScaler()
    df_escalado_local = pd.DataFrame(scaler.fit_transform(df_centro_local), columns=variables_matriz)
    
    # Matriz cuadrada usando Spearman
    matriz_local_insitu = df_escalado_local.corr(method='spearman')
    
    mostrar_cbar = True if idx == 2 else False
    
    # Dibujar el mapa de calor
    sns.heatmap(matriz_local_insitu, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f", ax=axes[idx],
                cbar=mostrar_cbar, annot_kws={"size": 11},
                cbar_kws={"shrink": .8, "label": "Coeficiente Rho de Spearman (In-Situ)"} if mostrar_cbar else None)
    
    axes[idx].set_title(nombres_centros[centro], fontsize=13, fontweight='bold', pad=15)
    axes[idx].set_xticklabels(variables_matriz, rotation=45, ha='right', fontsize=9)
    
    if idx == 0:
        axes[idx].set_yticklabels(variables_matriz, rotation=0, fontsize=9)
    else:
        axes[idx].set_yticklabels([])

plt.suptitle('Ground-Truthing: Correlación con Parámetros Ambientales Medidos In-Situ', 
             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
ruta_heatmap_insitu = os.path.join(carpeta_destino, "Heatmap_Correlacion_InSitu_Sernapesca.png")
plt.savefig(ruta_heatmap_insitu, bbox_inches='tight')
plt.show()

print(f"\nProceso completado. Nueva matriz guardada en: {ruta_heatmap_insitu}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_lag = pd.read_csv(ruta_dataset)
df_lag['Fecha'] = pd.to_datetime(df_lag['Fecha'])

df_lag = df_lag.sort_values(by=['id_centro', 'Fecha']).reset_index(drop=True)

print("Iniciando Cálculo de Correlación Cruzada Desfasada (Time Lags)...")

# ==========================================
# 2. Ingeniería de Características Base
# ==========================================
df_lag['Velocidad_Corriente_m_s'] = np.sqrt(df_lag['Corriente_E_m_s']**2 + df_lag['Corriente_N_m_s']**2)

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {110557: "Centro 110557 (Rojo)", 120069: "Centro 120069 (Azul)", 110745: "Centro 110745 (Verde)"}

variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m']
variable_objetivo = 'Hembras_Ovigeras'
max_lags = 12 

paleta_fisica = {
    'Temperatura_C': '#d62728',     # Rojo
    'Salinidad_psu': '#1f77b4',     # Azul
    'Capa_Mezcla_m': '#7f7f7f'      # Gris oscuro
}

# ==========================================
# 3. Cálculo Iterativo de Spearman Desfasado
# ==========================================
resultados_lags = []

for centro in centros_objetivo:
    df_centro = df_lag[df_lag['id_centro'] == centro].copy()
    
    for var in variables_fisicas:
        for lag in range(0, max_lags + 1):
          
            fisica_desfasada = df_centro[var].shift(lag)
            biologia_actual = df_centro[variable_objetivo]
            
      
            temp_df = pd.DataFrame({'Fisica': fisica_desfasada, 'Biologia': biologia_actual}).dropna()
            
            if len(temp_df) > 10:
                corr = temp_df['Fisica'].corr(temp_df['Biologia'], method='spearman')
            else:
                corr = np.nan
                
            resultados_lags.append({
                'id_centro': centro,
                'Variable_Fisica': var,
                'Lag': lag,
                'Correlacion_Spearman': corr
            })

df_resultados = pd.DataFrame(resultados_lags)

# ==========================================
# 4. Renderizado del Correlograma
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6), dpi=150, sharey=True)

for idx, centro in enumerate(centros_objetivo):
    data_plot = df_resultados[df_resultados['id_centro'] == centro]
    
    sns.lineplot(data=data_plot, x='Lag', y='Correlacion_Spearman', hue='Variable_Fisica',
                 ax=axes[idx], palette=paleta_fisica, marker='o', linewidth=2, markersize=8)
    

    axes[idx].axhline(0, color='black', linestyle='--', linewidth=1.5)
    
    axes[idx].set_title(nombres_centros[centro], fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Rezago Temporal (Semanas hacia el pasado)')
    axes[idx].set_xticks(range(0, max_lags + 1))
    
    if idx == 0:
        axes[idx].set_ylabel('Coeficiente de Correlación (Spearman)')
    else:
        axes[idx].set_ylabel('')
        

    if idx == 1:
        axes[idx].legend(title='Vector Oceanográfico', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    else:
        axes[idx].get_legend().remove()

plt.suptitle('Análisis de Correlación Cruzada (Time Lags): Impacto Físico sobre Hembras Ovígeras', 
             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
ruta_correlograma = os.path.join(carpeta_destino, "Correlograma_TimeLags.png")
plt.savefig(ruta_correlograma, bbox_inches='tight')
plt.show()

print(f"\nCálculo de rezagos temporales completado. Correlograma exportado a: {ruta_correlograma}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_lag = pd.read_csv(ruta_dataset)
df_lag['Fecha'] = pd.to_datetime(df_lag['Fecha'])

df_lag = df_lag.sort_values(by=['id_centro', 'Fecha']).reset_index(drop=True)

print("Iniciando Cálculo de Correlación Cruzada Expandido (Física + Biología)...")

# ==========================================
# 2. Ingeniería de Características y Paletas
# ==========================================
df_lag['Velocidad_Corriente_m_s'] = np.sqrt(df_lag['Corriente_E_m_s']**2 + df_lag['Corriente_N_m_s']**2)

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {110557: "Centro 110557 (Rojo)", 120069: "Centro 120069 (Azul)", 110745: "Centro 110745 (Verde)"}

variables_predictoras = [
    'Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 
    'Altura_Mar_m', 'Velocidad_Corriente_m_s',
    'Juveniles', 'Adultos_Moviles'
]
variable_objetivo = 'Hembras_Ovigeras'
max_lags = 12 # 3 meses de memoria

# Paleta distintiva para 7 variables
paleta_variables = {
    'Temperatura_C': '#d62728',           
    'Salinidad_psu': '#1f77b4',           
    'Capa_Mezcla_m': '#7f7f7f',          
    'Altura_Mar_m': '#9467bd',            
    'Velocidad_Corriente_m_s': '#8c564b', 
    'Juveniles': '#2ca02c',               
    'Adultos_Moviles': '#ff7f0e'          
}

# ==========================================
# 3. Cálculo Iterativo de Spearman Desfasado
# ==========================================
resultados_lags = []

for centro in centros_objetivo:
    df_centro = df_lag[df_lag['id_centro'] == centro].copy()
    
    for var in variables_predictoras:
        for lag in range(0, max_lags + 1):
            # Desplazamos la variable predictora hacia el futuro
            var_desfasada = df_centro[var].shift(lag)
            biologia_actual = df_centro[variable_objetivo]
            
            # DataFrame temporal para limpiar nulos generados por el shift o vacíos
            temp_df = pd.DataFrame({'Predictora': var_desfasada, 'Objetivo': biologia_actual}).dropna()
            
            if len(temp_df) > 10: 
                corr = temp_df['Predictora'].corr(temp_df['Objetivo'], method='spearman')
            else:
                corr = np.nan
                
            resultados_lags.append({
                'id_centro': centro,
                'Variable': var,
                'Lag': lag,
                'Correlacion_Spearman': corr
            })

df_resultados = pd.DataFrame(resultados_lags)

# ==========================================
# 4. Renderizado del Correlograma Expandido
# ==========================================
sns.set_theme(style="whitegrid")
# Aumentamos el tamaño para acomodar las 7 líneas con claridad
fig, axes = plt.subplots(1, 3, figsize=(22, 7), dpi=150, sharey=True)

for idx, centro in enumerate(centros_objetivo):
    data_plot = df_resultados[df_resultados['id_centro'] == centro]
    
    # Diferenciamos el grosor: Biología un poco más gruesa para que resalte
    sns.lineplot(data=data_plot, x='Lag', y='Correlacion_Spearman', hue='Variable',
                 ax=axes[idx], palette=paleta_variables, marker='o', 
                 linewidth=2.5, markersize=7, alpha=0.85)
    
    # Línea de referencia neutral
    axes[idx].axhline(0, color='black', linestyle='--', linewidth=1.5)
    
    axes[idx].set_title(nombres_centros[centro], fontsize=15, fontweight='bold')
    axes[idx].set_xlabel('Rezago Temporal (Semanas hacia el pasado)')
    axes[idx].set_xticks(range(0, max_lags + 1))
    
    if idx == 0:
        axes[idx].set_ylabel('Coeficiente de Correlación (Spearman)')
    else:
        axes[idx].set_ylabel('')
        
    # Limpieza de leyendas (solo dejamos una central abajo)
    axes[idx].get_legend().remove()

# Añadir una leyenda global en la parte inferior
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=7, title='Variables Predictoras (Físicas y Biológicas)', frameon=True)

plt.suptitle('Correlograma Expandido: Memoria Oceánica y Reloj Biológico sobre Hembras Ovígeras', 
             fontsize=18, fontweight='bold', y=1.03)

plt.tight_layout()
ruta_correlograma_exp = os.path.join(carpeta_destino, "Correlograma_Expandido_7Variables.png")
plt.savefig(ruta_correlograma_exp, bbox_inches='tight')
plt.show()

print(f"\nCorrelograma expandido completado. Imagen exportada a: {ruta_correlograma_exp}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_total = pd.read_csv(ruta_dataset)
col_centro = 'id_centro'

print("Iniciando Análisis de Carga Parasitaria Total e Individual...")

# ==========================================
# 2. Ingeniería de Características (Magnitud y Sumatoria)
# ==========================================
# Magnitud del vector de corriente
df_total['Velocidad_Corriente_m_s'] = np.sqrt(df_total['Corriente_E_m_s']**2 + df_total['Corriente_N_m_s']**2)

# Nueva Variable Objetivo: Sumatoria total de la carga parasitaria
df_total['Carga_Total_Parasitos'] = df_total['Juveniles'] + df_total['Adultos_Moviles'] + df_total['Hembras_Ovigeras']

# Definición del vector de variables para la matriz
variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_biologicas = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras', 'Carga_Total_Parasitos']
variables_matriz = variables_fisicas + variables_biologicas

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {110557: "Centro 110557 (Rojo)", 120069: "Centro 120069 (Azul)", 110745: "Centro 110745 (Verde)"}

# ==========================================
# 3. Renderizado de Matrices Cuadradas
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(26, 9), dpi=150)
sns.set_theme(style="white")

cmap = sns.diverging_palette(230, 20, as_cmap=True)

for idx, centro in enumerate(centros_objetivo):
    # Filtrado local por centro y remoción de filas con nulos
    df_centro_local = df_total[df_total[col_centro] == centro][variables_matriz].dropna().copy()
    
    # Estandarización de las escalas (Z-score)
    scaler = StandardScaler()
    df_escalado_local = pd.DataFrame(scaler.fit_transform(df_centro_local), columns=variables_matriz)
    
    # Cálculo del coeficiente de Spearman
    matriz_local = df_escalado_local.corr(method='spearman')
    
    # Control para mostrar la barra de color solo en el panel derecho
    mostrar_cbar = True if idx == 2 else False
    
    # Dibujar el Heatmap cuadrado completo
    sns.heatmap(matriz_local, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f", ax=axes[idx],
                cbar=mostrar_cbar, annot_kws={"size": 9},
                cbar_kws={"shrink": .8, "label": "Coeficiente Rho de Spearman"} if mostrar_cbar else None)
    
    axes[idx].set_title(nombres_centros[centro], fontsize=14, fontweight='bold', pad=15)
    axes[idx].set_xticklabels(variables_matriz, rotation=45, ha='right', fontsize=9)
    
    if idx == 0:
        axes[idx].set_yticklabels(variables_matriz, rotation=0, fontsize=9)
    else:
        axes[idx].set_yticklabels([])

plt.suptitle('Análisis de Interacción: Variables Físicas vs. Estadios Colectivos e Individuales del Parásito', 
             fontsize=18, fontweight='bold', y=1.05)

plt.tight_layout()
ruta_heatmap_total = os.path.join(carpeta_destino, "Heatmap_Correlacion_Carga_Total.png")
plt.savefig(ruta_heatmap_total, bbox_inches='tight')
plt.show()

print(f"\nMatrices generadas exitosamente. Imagen exportada a: {ruta_heatmap_total}")

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df_num = pd.read_csv(ruta_dataset)
df_num['Fecha'] = pd.to_datetime(df_num['Fecha'])

print("Iniciando Análisis Numérico Descriptivo...")

# ==========================================
# 2. Ingeniería de Características Base
# ==========================================
df_num['Año'] = df_num['Fecha'].dt.year
df_num['Mes'] = df_num['Fecha'].dt.month
df_num['Velocidad_Corriente_m_s'] = np.sqrt(df_num['Corriente_E_m_s']**2 + df_num['Corriente_N_m_s']**2)

def asignar_estacion(mes):
    if mes in [12, 1, 2]: return 'Verano'
    elif mes in [3, 4, 5]: return 'Otoño'
    elif mes in [6, 7, 8]: return 'Invierno'
    elif mes in [9, 10, 11]: return 'Primavera'

df_num['Estacion'] = df_num['Mes'].apply(asignar_estacion)

orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
df_num['Estacion'] = pd.Categorical(df_num['Estacion'], categories=orden_estaciones, ordered=True)

variables_analisis = [
    'Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 
    'Velocidad_Corriente_m_s', 'Hembras_Ovigeras'
]

agregaciones = {
    var: [
        ('Promedio', 'mean'), 
        ('Min', 'min'), 
        ('Max', 'max'), 
        ('Desv_Std', 'std')
    ] for var in variables_analisis
}

agg_dict = {key: [func[1] for func in value] for key, value in agregaciones.items()}

# Diccionario traductor para los nombres de las columnas
traductor_sufijos = {'mean': 'Prom', 'min': 'Min', 'max': 'Max', 'std': 'Std'}

# ==========================================
# 3. Cálculo de Estadísticos (Corregido)
# ==========================================
# A. Reporte General (Todo el periodo)
resumen_general = df_num.groupby(['id_centro', 'Estacion'], observed=False).agg(agg_dict)
# Aplanamos mapeando directamente el nombre de la función interna
resumen_general.columns = [f"{col[0]}_{traductor_sufijos[col[1]]}" for col in resumen_general.columns]
resumen_general = resumen_general.round(2)

# B. Reporte Anual (Desglosado por Año)
resumen_anual = df_num.groupby(['id_centro', 'Año', 'Estacion'], observed=False).agg(agg_dict)
resumen_anual.columns = [f"{col[0]}_{traductor_sufijos[col[1]]}" for col in resumen_anual.columns]
resumen_anual = resumen_anual.round(2)

# ==========================================
# 4. Exportación a Excel
# ==========================================
ruta_excel = os.path.join(carpeta_destino, "Analisis_Estadistico_Descriptivo.xlsx")

with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    resumen_general.to_excel(writer, sheet_name='Resumen_General')
    resumen_anual.to_excel(writer, sheet_name='Desglose_Anual')

print(f"\n¡Análisis numérico completado! Tablas exportadas exitosamente a: {ruta_excel}")

print("\n--- Muestra del Resumen General (Primeras filas) ---")
print(resumen_general.head(8))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
# Cargamos el dataset que ya trae las estaciones y las corrientes
ruta_input = "data/Dataset_Maestro_Estacional_ML.csv"
ruta_output = "data/Dataset_Maestro_Amplificado_ML.csv"

print("Iniciando Amplificación del Dataset...")
df = pd.read_csv(ruta_input)

# ==========================================
# 2. Ingeniería de Características (Nuevas Columnas)
# ==========================================

# A. Añadir la Región por Centro
# Ajusta los nombres de las regiones según la ubicación real de tus 3 centros
diccionario_regiones = {
    110557: "Aysen", 
    120069: "Magallanes",     
    110745: "Aysen"  
}
df['Region'] = df['id_centro'].map(diccionario_regiones)

# B. Sumatoria: Carga Total de Parásitos
# Consolida la biomasa total en la columna de agua
df['Carga_Total_Parasitos'] = df['Juveniles'] + df['Adultos_Moviles'] + df['Hembras_Ovigeras']

# C. Creación del Umbral/Etiqueta Normativa para Hembras Ovígeras
# Usamos np.select para categorizar de forma limpia y continua
condiciones = [
    (df['Hembras_Ovigeras'] < 1.0),                                    # Riesgo Bajo
    (df['Hembras_Ovigeras'] >= 1.0) & (df['Hembras_Ovigeras'] < 3.0),  # Riesgo Medio
    (df['Hembras_Ovigeras'] >= 3.0)                                    # Riesgo Alto (CAD)
]

etiquetas_riesgo = ['Bajo', 'Medio', 'Alto']

df['Nivel_Riesgo_HO'] = np.select(condiciones, etiquetas_riesgo, default='Desconocido')

# ==========================================
# 3. Validación y Exportación
# ==========================================
# Reordenar las columnas para que las nuevas queden visibles y lógicas
columnas_finales = [
    'id_centro', 'Region', 'Fecha', 'Año', 'Mes', 'Estacion', 
    'Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 
    'Corriente_E_m_s', 'Corriente_N_m_s', 'Velocidad_Corriente_m_s',
    'Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras', 'Carga_Total_Parasitos',
    'Nivel_Riesgo_HO'
]

# Si el dataset original tiene otras columnas, nos aseguramos de no perderlas
columnas_existentes = [col for col in columnas_finales if col in df.columns]
df_final = df[columnas_existentes]

df_final.to_csv(ruta_output, index=False)

print(f"¡Proceso completado exitosamente! Dataset guardado en: {ruta_output}")

# Mostrar una muestra de cómo quedaron las nuevas columnas
print("\n--- Muestra de la Matriz Amplificada ---")
print(df_final[['id_centro', 'Region', 'Hembras_Ovigeras', 'Carga_Total_Parasitos', 'Nivel_Riesgo_HO']].head(10))

# Imprimir un resumen de cuántos registros cayeron en cada categoría
print("\n--- Distribución del Nivel de Riesgo (Hembras Ovígeras) ---")
print(df_final['Nivel_Riesgo_HO'].value_counts())

***AAAAAAAAAAAAAAAAAAAAAH. FIXED ORDEN DE ESTACIONES***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Amplificado_ML.csv"
carpeta_destino = "Resultados_Exploracion/Boxplot_Estacional/"
os.makedirs(carpeta_destino, exist_ok=True)

print("Cargando Dataset Amplificado para generar gráficos individuales...")
df_est = pd.read_csv(ruta_dataset)

col_centro = 'id_centro'

# ==========================================
# 2. Preparación de Categorías (Ordenamiento Visual)
# ==========================================
orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
df_est['Estacion'] = pd.Categorical(df_est['Estacion'], categories=orden_estaciones, ordered=True)

paleta_estaciones = {
    'Verano': '#d62728',     # Rojo
    'Otoño': '#ff7f0e',      # Naranja
    'Invierno': '#1f77b4',   # Azul
    'Primavera': '#2ca02c'   # Verde
}

# ==========================================
# 3. Configuración de los Gráficos Individuales
# ==========================================
# Diccionario con las propiedades específicas de cada variable
config_graficos = [
    {
        'variable': 'Temperatura_C', 'titulo': 'Perfil Térmico Estacional (Ajuste Astronómico)', 
        'ylabel': 'Temperatura (°C)', 'invertir_y': False, 'min_cero': False, 'umbral': None
    },
    {
        'variable': 'Capa_Mezcla_m', 'titulo': 'Profundidad de la Capa de Mezcla Estacional', 
        'ylabel': 'Profundidad (m)', 'invertir_y': True, 'min_cero': False, 'umbral': None
    },
    {
        'variable': 'Hembras_Ovigeras', 'titulo': 'Impacto Epidemiológico Estacional (Hembras Ovígeras)', 
        'ylabel': 'Cantidad Promedio', 'invertir_y': False, 'min_cero': True, 'umbral': 3
    },
    {
        'variable': 'Salinidad_psu', 'titulo': 'Perfil de Salinidad Estacional por Centro', 
        'ylabel': 'Salinidad (psu)', 'invertir_y': False, 'min_cero': False, 'umbral': None
    },
    {
        'variable': 'Velocidad_Corriente_m_s', 'titulo': 'Evolución Estacional de Velocidad de Corriente', 
        'ylabel': 'Velocidad (m/s)', 'invertir_y': False, 'min_cero': True, 'umbral': None
    },
    {
        'variable': 'Altura_Mar_m', 'titulo': 'Fluctuación Estacional de Altura del Mar', 
        'ylabel': 'Altura del Mar (m)', 'invertir_y': False, 'min_cero': False, 'umbral': None
    },
    {
        'variable': 'Juveniles', 'titulo': 'Carga Estacional de Juveniles por Centro', 
        'ylabel': 'Cantidad Promedio', 'invertir_y': False, 'min_cero': True, 'umbral': None
    },
    {
        'variable': 'Adultos_Moviles', 'titulo': 'Carga Estacional de Adultos Móviles por Centro', 
        'ylabel': 'Cantidad Promedio', 'invertir_y': False, 'min_cero': True, 'umbral': None
    }
]

# ==========================================
# 4. Generación y Exportación de Boxplots
# ==========================================
sns.set_theme(style="whitegrid")

for config in config_graficos:
    plt.figure(figsize=(10, 6), dpi=150)
    
    # Renderizado del boxplot
    ax = sns.boxplot(
        x=col_centro, y=config['variable'], hue='Estacion', 
        data=df_est, palette=paleta_estaciones, width=0.7, fliersize=2
    )
    
    # Personalización de títulos y etiquetas
    plt.title(config['titulo'], fontweight='bold', fontsize=14, pad=15)
    plt.xlabel('Centro de Cultivo', fontweight='bold')
    plt.ylabel(config['ylabel'], fontweight='bold')
    
    # Lógicas condicionales según el diccionario
    if config['invertir_y']:
        plt.gca().invert_yaxis()
        
    if config['min_cero']:
        plt.ylim(bottom=0)
        
    if config['umbral'] is not None:
        plt.axhline(y=config['umbral'], color='black', linestyle='--', alpha=0.7, label=f"Umbral Crítico ({config['umbral']})")
        
    # Posicionar la leyenda fuera del gráfico para que no tape los datos
    plt.legend(title='Estación', bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    
    # Ajuste fino y guardado
    plt.tight_layout()
    nombre_archivo = f"Boxplot_{config['variable']}.png"
    ruta_guardado = os.path.join(carpeta_destino, nombre_archivo)
    
    plt.savefig(ruta_guardado, bbox_inches='tight')
    plt.close() # Cierra la figura para liberar memoria antes de la siguiente iteración
    
    print(f"Gráfico exportado: {nombre_archivo}")

print(f"\nProceso finalizado. Las 8 imágenes se encuentran en la carpeta '{carpeta_destino}'")

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_input = "data/Dataset_Maestro_Global_ML.csv"
ruta_output = "data/Dataset_Maestro_Amplificado_ML.csv"

print("Iniciando Pipeline de Preprocesamiento Maestro (Sin pérdida de datos)...")
df = pd.read_csv(ruta_input)

# Control de seguridad para el nombre de la columna semana
col_semana = 'Semana' if 'Semana' in df.columns else 'semana'

# ==========================================
# 2. INGENIERÍA DE CARACTERÍSTICAS (Feature Engineering)
# ==========================================

# A. Región por Centro
diccionario_regiones = {
    110557: "Aysen", 
    120069: "Magallanes",     
    110745: "Aysen"  
}
df['Region'] = df['id_centro'].map(diccionario_regiones)

# B. Magnitud de Velocidad de Corriente
df['Velocidad_Corriente_m_s'] = np.sqrt(df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2)

# C. Estaciones basadas en Solsticios y Equinoccios (Hemisferio Sur)
def asignar_estacion_astronomica(sem):
    if 1 <= sem <= 11 or 51 <= sem <= 53:
        return 'Verano'
    elif 12 <= sem <= 24:
        return 'Otoño'
    elif 25 <= sem <= 37:
        return 'Invierno'
    elif 38 <= sem <= 50:
        return 'Primavera'
    else:
        return 'Desconocido'

df['Estacion'] = df[col_semana].apply(asignar_estacion_astronomica)

# D. Sumatoria Total de Parásitos (Mantiene los vacíos matemáticos intactos)
df['Total_Parasitos_'] = df['Juveniles'] + df['Adultos_Moviles'] + df['Hembras_Ovigeras']

# E. Categorización de Riesgo (Target Predictivo)
condiciones = [
    (df['Hembras_Ovigeras'] < 1.0),                                    
    (df['Hembras_Ovigeras'] >= 1.0) & (df['Hembras_Ovigeras'] < 3.0),  
    (df['Hembras_Ovigeras'] >= 3.0)                                    
]
etiquetas_riesgo = ['Bajo', 'Medio', 'Alto']

# Se asigna 'Desconocido' a todos los registros que no tienen conteo parasitario
df['Nivel_Riesgo_HO'] = np.select(condiciones, etiquetas_riesgo, default='Desconocido')

# ==========================================
# 3. Estructuración Final y Exportación
# ==========================================
columnas_finales = [
    'id_centro', 'Region', 'Fecha', 'Año', 'Mes', 'Dia', col_semana, 'Estacion', 
    'Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 
    'Corriente_E_m_s', 'Corriente_N_m_s', 'Velocidad_Corriente_m_s',
    'Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras', 'Total_Parasitos_',
    'Nivel_Riesgo_HO'
]

columnas_existentes = [col for col in columnas_finales if col in df.columns]
df_final = df[columnas_existentes]

df_final.to_csv(ruta_output, index=False)

print(f"¡Matriz amplificada guardada con el 100% de los registros en: {ruta_output}")
print("\n--- Distribución del Nivel de Riesgo (Incluyendo vacíos operativos) ---")
print(df_final['Nivel_Riesgo_HO'].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ==========================================
# 1. Configuración Inicial y Carga de Datos
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Amplificado_ML.csv"
carpeta_destino = "Resultados_Exploracion/Series_Tiempo_Estacional/"
os.makedirs(carpeta_destino, exist_ok=True)

print("Cargando Dataset Amplificado para generar series de tiempo individuales...")
df_ts = pd.read_csv(ruta_dataset)
df_ts['Fecha'] = pd.to_datetime(df_ts['Fecha'])

col_centro = 'id_centro'

# ==========================================
# 2. Configuración Visual (Paletas y Fondos)
# ==========================================
sns.set_theme(style="white")

# Paleta de centros
paleta_centros = {
    110557: '#d62728',  # Rojo
    120069: '#5fa2ce',  # Azul claro
    110745: '#2ca02c'   # Verde
}

# Colores de fondo por estación
color_fondo_estacion = {
    'Verano': "#EE291B",    
    'Otoño': "#DBB433",     
    'Invierno': "#3F92EB",  
    'Primavera': "#67F052"  
}

fechas_unicas = df_ts[['Fecha', 'Estacion']].drop_duplicates().sort_values('Fecha')

# ==========================================
# 3. Configuración de los Gráficos Individuales
# ==========================================
# Respetando los títulos asignados para cada variable
config_graficos = [
    {'var': 'Temperatura_C', 'titulo': 'Temperatura (°C)', 'invertir_y': False, 'umbral': None},
    {'var': 'Capa_Mezcla_m', 'titulo': 'Capa de Mezcla (m)', 'invertir_y': True, 'umbral': None},
    {'var': 'Hembras_Ovigeras', 'titulo': 'Promedio de Hembras Ovígeras', 'invertir_y': False, 'umbral': 3},
    {'var': 'Salinidad_psu', 'titulo': 'Salinidad (psu)', 'invertir_y': False, 'umbral': None},
    {'var': 'Velocidad_Corriente_m_s', 'titulo': 'Magnitud de Corriente Hidrodinámica (m/s)', 'invertir_y': False, 'umbral': None},
    {'var': 'Altura_Mar_m', 'titulo': 'Fluctuación Altura del Mar (m)', 'invertir_y': False, 'umbral': None},
    {'var': 'Juveniles', 'titulo': 'Promedio de Juveniles', 'invertir_y': False, 'umbral': None},
    {'var': 'Adultos_Moviles', 'titulo': 'Promedio de Adultos Móviles', 'invertir_y': False, 'umbral': None}
]

# Construcción de la leyenda global que se aplicará a todos los gráficos
leyenda_centros = [Line2D([0], [0], color=paleta_centros[c], lw=2, label=f'Centro {c}') for c in paleta_centros]
leyenda_estaciones = [Patch(facecolor=color_fondo_estacion[est], label=est, alpha=0.3) for est in color_fondo_estacion]

# ==========================================
# 4. Generación y Exportación de Series de Tiempo
# ==========================================
for config in config_graficos:
    plt.figure(figsize=(12, 5), dpi=150)
    ax = plt.gca()
    
    # A. Trazar líneas
    sns.lineplot(x='Fecha', y=config['var'], hue=col_centro, data=df_ts, 
                 palette=paleta_centros, ax=ax, linewidth=1.5, alpha=0.8)
    
    # B. Aplicar sombreado estacional
    for estacion, color in color_fondo_estacion.items():
        ax.fill_between(fechas_unicas['Fecha'], 0, 1, 
                        where=(fechas_unicas['Estacion'] == estacion), 
                        color=color, alpha=0.15, transform=ax.get_xaxis_transform())

    # C. Configuraciones de ejes y títulos
    plt.title(config['titulo'], fontweight='bold', fontsize=14, pad=15)
    plt.xlabel('Fecha de Medición', fontweight='bold')
    plt.ylabel(config['var'], fontweight='bold')
    
    if df_ts[config['var']].min() >= 0:
        plt.ylim(bottom=0)
        
    if config['invertir_y']:
        plt.gca().invert_yaxis()
        
    # D. Gestión de Leyendas Unificadas
    handles_leyenda = leyenda_centros + leyenda_estaciones
    
    if config['umbral'] is not None:
        plt.axhline(y=config['umbral'], color='black', linestyle='--', linewidth=2)
        handles_leyenda.append(Line2D([0], [0], color='black', linestyle='--', lw=2, label=f'Umbral Crítico ({config["umbral"]})'))
    
    
    ax.get_legend().remove()
    plt.legend(handles=handles_leyenda, loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0., title='Leyenda Global')
    
    # E. Guardado
    plt.tight_layout()
    nombre_archivo = f"TS_Estacional_{config['var']}.png"
    ruta_guardado = os.path.join(carpeta_destino, nombre_archivo)
    
    plt.savefig(ruta_guardado, bbox_inches='tight')
    plt.close()
    
    print(f"Gráfico exportado: {nombre_archivo}")

print(f"\nProceso finalizado. Las 8 series de tiempo se encuentran en la carpeta '{carpeta_destino}'")

In [ ]:
"""
PUNTO 2 - CORRECCIÓN: Eliminación del StandardScaler en matrices Spearman
=========================================================================
Spearman opera sobre rangos internamente, por lo que estandarizar con Z-score
antes del cálculo no tiene efecto matemático pero sí introduce confusión metodológica.
Se elimina el StandardScaler de todas las celdas de heatmap.
También se corrigen los títulos de los gráficos a formato técnico-descriptivo.
"""

# ============================================================
# CELDA A: Heatmap Satélite (Dataset_Maestro_Global_ML.csv)
# Variables: Físicas satelitales + 3 estadios parasitarios
# Reemplaza la celda que usa: "Heatmap_Correlacion_Cuadrado_Centros.png"
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_destino = "Resultados_Exploracion/HeatMap/"
os.makedirs(carpeta_destino, exist_ok=True)

df_corr = pd.read_csv(ruta_dataset)
col_centro = 'id_centro'

df_corr['Velocidad_Corriente_m_s'] = np.sqrt(
    df_corr['Corriente_E_m_s']**2 + df_corr['Corriente_N_m_s']**2
)

variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m',
                     'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_biologicas = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
variables_matriz = variables_fisicas + variables_biologicas

# Etiquetas legibles para los ejes
etiquetas_ejes = ['Temperatura (°C)', 'Salinidad (psu)', 'Capa de Mezcla (m)',
                  'Altura del Mar (m)', 'Vel. Corriente (m/s)',
                  'Juveniles', 'Adultos Móviles', 'Hembras Ovígeras']

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {
    110557: "Centro 110557",
    120069: "Centro 120069",
    110745: "Centro 110745"
}
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.set_theme(style="white")


fig, axes = plt.subplots(1, 3, figsize=(24, 8), dpi=150)

for idx, centro in enumerate(centros_objetivo):
    df_centro = df_corr[df_corr[col_centro] == centro][variables_matriz].dropna()

    
    matriz = df_centro.corr(method='spearman')

    mostrar_cbar = idx == 2
    sns.heatmap(
        matriz, cmap=cmap, vmax=1, vmin=-1, center=0,
        square=True, linewidths=0.5, annot=True, fmt=".2f",
        ax=axes[idx], cbar=mostrar_cbar,
        annot_kws={"size": 10},
        cbar_kws={"shrink": 0.8, "label": "Rho de Spearman"} if mostrar_cbar else None
    )

    axes[idx].set_title(nombres_centros[centro], fontsize=14, fontweight='bold', pad=12)
    axes[idx].set_xticklabels(etiquetas_ejes, rotation=45, ha='right', fontsize=9)
    if idx == 0:
        axes[idx].set_yticklabels(etiquetas_ejes, rotation=0, fontsize=9)
    else:
        axes[idx].set_yticklabels([])


plt.suptitle(
    'Correlación de Spearman entre Variables Físicas y Estadios Parasitarios por Centro',
    fontsize=16, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.savefig(os.path.join(carpeta_destino, "Heatmap_Combinado.png"), bbox_inches='tight')
plt.close()
print("Heatmap combinado exportado: Heatmap_Combinado.png")


for centro in centros_objetivo:
    plt.figure(figsize=(9, 7), dpi=150)
    df_centro = df_corr[df_corr[col_centro] == centro][variables_matriz].dropna()

    
    matriz = df_centro.corr(method='spearman')

    sns.heatmap(
        matriz, cmap=cmap, vmax=1, vmin=-1, center=0,
        square=True, linewidths=0.5, annot=True, fmt=".2f",
        cbar=True, annot_kws={"size": 11},
        cbar_kws={"shrink": 0.8, "label": "Rho de Spearman"}
    )
    plt.title(
        f'Correlación de Spearman — {nombres_centros[centro]}',
        fontsize=14, fontweight='bold', pad=12
    )
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.savefig(
        os.path.join(carpeta_destino, f"Heatmap_Individual_{centro}.png"),
        bbox_inches='tight'
    )
    plt.close()
    print(f"Heatmap individual exportado: Heatmap_Individual_{centro}.png")


# ============================================================
# CELDA B: Heatmap Amplificado (Dataset_Maestro_Amplificado_ML.csv)
# Variables: Físicas + Biológicas + Estación numérica + Nivel de Riesgo
# ============================================================

ruta_dataset_amp = "data/Dataset_Maestro_Amplificado_ML.csv"
df_corr_amp = pd.read_csv(ruta_dataset_amp)

# Excluir periodos de descanso (sin datos parasitarios) para el análisis de correlación
df_valido = df_corr_amp[df_corr_amp['Nivel_Riesgo_HO'] != 'Desconocido'].copy()

# Codificación ordinal
mapa_riesgo = {'Bajo': 1, 'Medio': 2, 'Alto': 3}
mapa_estacion = {'Verano': 1, 'Otoño': 2, 'Invierno': 3, 'Primavera': 4}
df_valido['Riesgo_Numerico'] = df_valido['Nivel_Riesgo_HO'].map(mapa_riesgo)
df_valido['Estacion_Numerica'] = df_valido['Estacion'].map(mapa_estacion)

variables_amp = (variables_fisicas + variables_biologicas +
                 ['Estacion_Numerica', 'Riesgo_Numerico'])

etiquetas_amp = (etiquetas_ejes +
                 ['Estación (ord.)', 'Nivel de Riesgo (ord.)'])

# --- Panel combinado amplificado ---
fig, axes = plt.subplots(1, 3, figsize=(26, 9), dpi=150)

for idx, centro in enumerate(centros_objetivo):
    df_centro = df_valido[df_valido[col_centro] == centro][variables_amp].dropna()

    
    matriz = df_centro.corr(method='spearman')

    mostrar_cbar = idx == 2
    sns.heatmap(
        matriz, cmap=cmap, vmax=1, vmin=-1, center=0,
        square=True, linewidths=0.5, annot=True, fmt=".2f",
        ax=axes[idx], cbar=mostrar_cbar,
        annot_kws={"size": 9},
        cbar_kws={"shrink": 0.8, "label": "Rho de Spearman"} if mostrar_cbar else None
    )
    axes[idx].set_title(nombres_centros[centro], fontsize=14, fontweight='bold', pad=12)
    axes[idx].set_xticklabels(etiquetas_amp, rotation=45, ha='right', fontsize=9)
    if idx == 0:
        axes[idx].set_yticklabels(etiquetas_amp, rotation=0, fontsize=9)
    else:
        axes[idx].set_yticklabels([])

plt.suptitle(
    'Correlación de Spearman: Variables Físicas, Biológicas, Estacionalidad y Nivel de Riesgo',
    fontsize=16, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.savefig(
    os.path.join(carpeta_destino, "Heatmap_Combinado_Amplificado.png"),
    bbox_inches='tight'
)
plt.close()
print("Heatmap combinado amplificado exportado: Heatmap_Combinado_Amplificado.png")


for centro in centros_objetivo:
    plt.figure(figsize=(10, 8), dpi=150)
    df_centro = df_valido[df_valido[col_centro] == centro][variables_amp].dropna()

    
    matriz = df_centro.corr(method='spearman')

    sns.heatmap(
        matriz, cmap=cmap, vmax=1, vmin=-1, center=0,
        square=True, linewidths=0.5, annot=True, fmt=".2f",
        cbar=True, annot_kws={"size": 10},
        cbar_kws={"shrink": 0.8, "label": "Rho de Spearman"}
    )
    plt.title(
        f'Correlación de Spearman (variables amplificadas) — {nombres_centros[centro]}',
        fontsize=13, fontweight='bold', pad=12
    )
    plt.xticks(
        ticks=np.arange(len(etiquetas_amp)) + 0.5,
        labels=etiquetas_amp, rotation=45, ha='right', fontsize=10
    )
    plt.yticks(
        ticks=np.arange(len(etiquetas_amp)) + 0.5,
        labels=etiquetas_amp, rotation=0, fontsize=10
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(carpeta_destino, f"Heatmap_Individual_Amplificado_{centro}.png"),
        bbox_inches='tight'
    )
    plt.close()
    print(f"Heatmap individual amplificado exportado: Heatmap_Individual_Amplificado_{centro}.png")

print("\nPunto 2 completado. Todos los heatmaps en: Resultados_Exploracion/HeatMap/")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Amplificado_ML.csv"
carpeta_destino = "Resultados_Exploracion/HeatMap/"
os.makedirs(carpeta_destino, exist_ok=True)

df_corr = pd.read_csv(ruta_dataset)
col_centro = 'id_centro'

print("Generando Matrices Cuadradas con Variables Amplificadas...")

# ==========================================
# 2. Ingeniería de Características (Codificación Ordinal)
# ==========================================
# A. Filtramos en memoria los 'Desconocidos' para que la matemática funcione
df_valido = df_corr[df_corr['Nivel_Riesgo_HO'] != 'Desconocido'].copy()

# B. Transformamos las etiquetas de texto a jerarquías numéricas
mapa_riesgo = {'Bajo': 1, 'Medio': 2, 'Alto': 3}
mapa_estacion = {'Verano': 1, 'Otoño': 2, 'Invierno': 3, 'Primavera': 4}

df_valido['Riesgo_Numerico'] = df_valido['Nivel_Riesgo_HO'].map(mapa_riesgo)
df_valido['Estacion_Numerica'] = df_valido['Estacion'].map(mapa_estacion)

# C. Definición de variables para la matriz
variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_biologicas = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
variables_nuevas = ['Estacion_Numerica', 'Riesgo_Numerico']

variables_matriz = variables_fisicas + variables_biologicas + variables_nuevas

# Nombres legibles para los ejes del gráfico
nombres_ejes = ['Temperatura', 'Salinidad', 'Capa Mezcla', 'Altura Mar', 'Vel. Corriente', 
                'Juveniles', 'Adultos Móv.', 'Hembras Ov.', 'Estación (1-4)', 'Nivel Riesgo (1-3)']

centros_objetivo = [110557, 120069, 110745]
nombres_centros = {110557: "Centro 110557 (Rojo)", 120069: "Centro 120069 (Azul)", 110745: "Centro 110745 (Verde)"}
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# ==========================================
# 3. Renderizado de Panel Combinado (3 en 1)
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(26, 9), dpi=150)
sns.set_theme(style="white")

for idx, centro in enumerate(centros_objetivo):
    # Filtrado estricto por centro
    df_centro_local = df_valido[df_valido[col_centro] == centro][variables_matriz].dropna().copy()
    
    # Estandarización local
    scaler = StandardScaler()
    df_escalado_local = pd.DataFrame(scaler.fit_transform(df_centro_local), columns=variables_matriz)
    
    # Cálculo de Spearman
    matriz_local = df_escalado_local.corr(method='spearman')
    
    mostrar_cbar = True if idx == 2 else False
    
    # Dibujar el Heatmap en el panel correspondiente
    sns.heatmap(matriz_local, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f", ax=axes[idx],
                cbar=mostrar_cbar, annot_kws={"size": 10},
                cbar_kws={"shrink": .8, "label": "Coeficiente Rho de Spearman"} if mostrar_cbar else None)
    
    axes[idx].set_title(nombres_centros[centro], fontsize=15, fontweight='bold', pad=15)
    axes[idx].set_xticklabels(nombres_ejes, rotation=45, ha='right', fontsize=11)
    
    if idx == 0:
        axes[idx].set_yticklabels(nombres_ejes, rotation=0, fontsize=11)
    else:
        axes[idx].set_yticklabels([]) # Limpiamos el eje Y para no saturar

plt.suptitle('Interacción Física-Biología incluyendo Nivel de Riesgo y Estacionalidad', 
             fontsize=18, fontweight='bold', y=1.05)

plt.tight_layout()
# NUEVO NOMBRE DE ARCHIVO
ruta_combinada = os.path.join(carpeta_destino, "Heatmap_Combinado_Amplificado_NuevasVar.png")
plt.savefig(ruta_combinada, bbox_inches='tight')
plt.close() 
print(f"Gráfico combinado exportado en: {ruta_combinada}")

# ==========================================
# 4. Renderizado de Gráficos Individuales
# ==========================================
for centro in centros_objetivo:
    plt.figure(figsize=(11, 9), dpi=150)
    
    df_centro_local = df_valido[df_valido[col_centro] == centro][variables_matriz].dropna().copy()
    scaler = StandardScaler()
    df_escalado_local = pd.DataFrame(scaler.fit_transform(df_centro_local), columns=variables_matriz)
    matriz_local = df_escalado_local.corr(method='spearman')
    
    sns.heatmap(matriz_local, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, annot=True, fmt=".2f",
                cbar=True, annot_kws={"size": 11},
                cbar_kws={"shrink": .8, "label": "Coeficiente Rho de Spearman"})
    
    plt.title(f"Matriz de Correlación Amplificada - {nombres_centros[centro]}", fontsize=15, fontweight='bold', pad=15)
    
    plt.xticks(ticks=np.arange(len(nombres_ejes)) + 0.5, labels=nombres_ejes, rotation=45, ha='right', fontsize=11)
    plt.yticks(ticks=np.arange(len(nombres_ejes)) + 0.5, labels=nombres_ejes, rotation=0, fontsize=11)
    
    plt.tight_layout()
    # NUEVO NOMBRE DE ARCHIVO
    nombre_archivo = f"Heatmap_Individual_Amplificado_NuevasVar_{centro}.png"
    ruta_individual = os.path.join(carpeta_destino, nombre_archivo)
    
    plt.savefig(ruta_individual, bbox_inches='tight')
    plt.close() 
    print(f"Gráfico individual exportado: {nombre_archivo}")

print(f"\nProceso finalizado. Las imágenes están en '{carpeta_destino}' sin sobreescribir las anteriores.")

In [ ]:
"""
PUNTO 3 - Correlograma de Rezagos Temporales con p-values corregidos
=====================================================================
Cambios respecto al código original:
- Se reemplaza pandas .corr() por scipy.stats.spearmanr() para obtener p-values
- Se aplica corrección de Benjamini-Hochberg sobre los 117 tests simultáneos
  (13 lags × 3 variables físicas × 3 centros)
- Los puntos significativos (p_corr < 0.05) se marcan con marcador relleno
- Los no significativos se muestran con marcador vacío (mismo color, sin relleno)
- Se exporta tabla de resultados completa a Resultados_Exploracion/Estadisticas/
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"
carpeta_graficos = "Resultados_Exploracion/HeatMap/"         # Correlograma va aquí (consistente con tu estructura)
carpeta_stats   = "Resultados_Exploracion/Estadisticas/"     # Tabla de resultados aquí
os.makedirs(carpeta_graficos, exist_ok=True)
os.makedirs(carpeta_stats, exist_ok=True)

df_lag = pd.read_csv(ruta_dataset)
df_lag['Fecha'] = pd.to_datetime(df_lag['Fecha'])
df_lag = df_lag.sort_values(by=['id_centro', 'Fecha']).reset_index(drop=True)

print("Iniciando cálculo de correlación cruzada con p-values (Spearman + BH)...")

# ==========================================
# 2. Configuración del análisis
# ==========================================
df_lag['Velocidad_Corriente_m_s'] = np.sqrt(
    df_lag['Corriente_E_m_s']**2 + df_lag['Corriente_N_m_s']**2
)

centros_objetivo  = [110557, 120069, 110745]
nombres_centros   = {
    110557: "Centro 110557",
    120069: "Centro 120069",
    110745: "Centro 110745"
}

variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m']
variable_objetivo = 'Hembras_Ovigeras'
max_lags          = 12

paleta_fisica = {
    'Temperatura_C': '#d62728',
    'Salinidad_psu': '#1f77b4',
    'Capa_Mezcla_m': '#7f7f7f'
}

# ==========================================
# 3. Cálculo de Spearman con p-value por lag
# ==========================================
# Usamos scipy.stats.spearmanr() en lugar de pandas .corr()
# porque pandas no devuelve el p-value directamente.
resultados = []

for centro in centros_objetivo:
    df_centro = df_lag[df_lag['id_centro'] == centro].copy()

    for var in variables_fisicas:
        for lag in range(0, max_lags + 1):

            fisica_desfasada = df_centro[var].shift(lag)
            biologia_actual  = df_centro[variable_objetivo]

            temp_df = pd.DataFrame({
                'Fisica'  : fisica_desfasada,
                'Biologia': biologia_actual
            }).dropna()

            if len(temp_df) > 10:
                rho, pval = stats.spearmanr(temp_df['Fisica'], temp_df['Biologia'])
            else:
                rho, pval = np.nan, np.nan

            resultados.append({
                'id_centro'        : centro,
                'Variable_Fisica'  : var,
                'Lag'              : lag,
                'n'                : len(temp_df),
                'Rho_Spearman'     : round(rho, 4) if not np.isnan(rho) else np.nan,
                'p_valor_raw'      : round(pval, 6) if not np.isnan(pval) else np.nan
            })

df_resultados = pd.DataFrame(resultados)

# ==========================================
# 4. Corrección de Benjamini-Hochberg
# ==========================================
# Se aplica la corrección sobre TODOS los tests válidos simultáneamente
# (no por centro ni por variable por separado), lo que es el enfoque más riguroso.
mask_validos = df_resultados['p_valor_raw'].notna()

_, p_corregidos, _, _ = multipletests(
    df_resultados.loc[mask_validos, 'p_valor_raw'],
    alpha=0.05,
    method='fdr_bh'     # Benjamini-Hochberg
)

df_resultados.loc[mask_validos, 'p_valor_BH']    = p_corregidos.round(6)
df_resultados['Significativo_BH'] = df_resultados['p_valor_BH'] < 0.05

# ==========================================
# 5. Exportación de la tabla de resultados
# ==========================================
ruta_tabla = os.path.join(carpeta_stats, "Timelags_Spearman_pvalores.csv")
df_resultados.to_csv(ruta_tabla, index=False)
print(f"Tabla de resultados exportada: {ruta_tabla}")
print(f"\nTests significativos (p_BH < 0.05): "
      f"{df_resultados['Significativo_BH'].sum()} de {mask_validos.sum()} tests válidos\n")

# ==========================================
# 6. Correlograma con marcadores de significancia
# ==========================================
# Convención visual:
#   Marcador RELLENO  → p_BH < 0.05 (significativo)
#   Marcador VACÍO    → p_BH ≥ 0.05 (no significativo)
# Se mantiene el mismo color por variable para no romper la paleta existente.

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6), dpi=150, sharey=True)

for idx, centro in enumerate(centros_objetivo):
    ax = axes[idx]
    data_centro = df_resultados[df_resultados['id_centro'] == centro]

    for var in variables_fisicas:
        data_var = data_centro[data_centro['Variable_Fisica'] == var].sort_values('Lag')
        color     = paleta_fisica[var]

        lags      = data_var['Lag'].values
        rhos      = data_var['Rho_Spearman'].values
        sig_mask  = data_var['Significativo_BH'].values

        # Línea de fondo continua
        ax.plot(lags, rhos, color=color, linewidth=2, alpha=0.7, label=var)

        # Puntos significativos: marcador relleno
        ax.scatter(
            lags[sig_mask], rhos[sig_mask],
            color=color, s=80, zorder=5,
            marker='o'
        )

        # Puntos NO significativos: marcador vacío (facecolor='none')
        ax.scatter(
            lags[~sig_mask], rhos[~sig_mask],
            facecolors='none', edgecolors=color, s=80, zorder=5,
            linewidths=1.5, marker='o'
        )

    # Línea de referencia en cero
    ax.axhline(0, color='black', linestyle='--', linewidth=1.2)

    # Banda de referencia ±0.2 (orientación visual, no es IC formal)
    ax.axhspan(-0.2, 0.2, color='grey', alpha=0.07)

    ax.set_title(nombres_centros[centro], fontsize=13, fontweight='bold')
    ax.set_xlabel('Rezago temporal (semanas)')
    ax.set_xticks(range(0, max_lags + 1))

    if idx == 0:
        ax.set_ylabel('Rho de Spearman')

    ax.get_legend().remove() if ax.get_legend() else None

# Leyenda unificada con explicación de marcadores
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

handles_vars = [
    Line2D([0], [0], color=paleta_fisica[v], lw=2, label=v.replace('_', ' '))
    for v in variables_fisicas
]
handles_sig = [
    Line2D([0], [0], marker='o', color='grey', lw=0,
           markerfacecolor='grey', markersize=8, label='p < 0.05 (BH)'),
    Line2D([0], [0], marker='o', color='grey', lw=0,
           markerfacecolor='none', markeredgecolor='grey',
           markeredgewidth=1.5, markersize=8, label='p ≥ 0.05 (BH)')
]

fig.legend(
    handles=handles_vars + handles_sig,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.08),
    ncol=5,
    title='Variable física  |  Significancia (Benjamini-Hochberg)',
    frameon=True
)

plt.suptitle(
    'Correlación de Spearman con Rezago Temporal: Variables Físicas vs. Hembras Ovígeras',
    fontsize=15, fontweight='bold', y=1.02
)

plt.tight_layout()
ruta_correlograma = os.path.join(carpeta_graficos, "Correlograma_TimeLags_pvalores.png")
plt.savefig(ruta_correlograma, bbox_inches='tight')
plt.close()
print(f"Correlograma exportado: {ruta_correlograma}")
print("\nPunto 3 completado.")

In [ ]:
"""
PUNTO 4 - Comparación entre Centros: Mann-Whitney U + Benjamini-Hochberg
=========================================================================
Compara la distribución de cada variable oceanográfica entre los tres centros
mediante pruebas pairwise Mann-Whitney U (no paramétrico, sin supuesto de normalidad).
Los p-values se corrigen con Benjamini-Hochberg sobre el total de comparaciones.

Para las variables oceanográficas se usan TODOS los registros disponibles
(incluyendo periodos sin datos parasitarios).

Outputs en Resultados_Exploracion/Estadisticas/:
  - MannWhitney_Comparacion_Centros.csv     → tabla completa de resultados
  - MannWhitney_Comparacion_Centros.png     → heatmap de significancia por variable
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import os

# ==========================================
# 1. Configuración
# ==========================================
ruta_dataset    = "data/Dataset_Maestro_Global_ML.csv"
carpeta_stats   = "Resultados_Exploracion/Estadisticas/"
os.makedirs(carpeta_stats, exist_ok=True)

df = pd.read_csv(ruta_dataset)

df['Velocidad_Corriente_m_s'] = np.sqrt(
    df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2
)

centros_objetivo = [110557, 120069, 110745]
nombres_centros  = {
    110557: "Centro\n110557",
    120069: "Centro\n120069",
    110745: "Centro\n110745"
}

variables_fisicas = [
    'Temperatura_C',
    'Salinidad_psu',
    'Capa_Mezcla_m',
    'Altura_Mar_m',
    'Velocidad_Corriente_m_s'
]

etiquetas_variables = {
    'Temperatura_C'          : 'Temperatura (°C)',
    'Salinidad_psu'          : 'Salinidad (psu)',
    'Capa_Mezcla_m'          : 'Capa de Mezcla (m)',
    'Altura_Mar_m'           : 'Altura del Mar (m)',
    'Velocidad_Corriente_m_s': 'Vel. Corriente (m/s)'
}

pares = list(combinations(centros_objetivo, 2))  # (110557,120069), (110557,110745), (120069,110745)

print("Iniciando comparación Mann-Whitney U entre centros...\n")

# ==========================================
# 2. Cálculo de Mann-Whitney U por variable y par de centros
# ==========================================
resultados = []

for var in variables_fisicas:
    for (c1, c2) in pares:

        datos_c1 = df[df['id_centro'] == c1][var].dropna().values
        datos_c2 = df[df['id_centro'] == c2][var].dropna().values

        if len(datos_c1) > 5 and len(datos_c2) > 5:
            stat, pval = stats.mannwhitneyu(
                datos_c1, datos_c2,
                alternative='two-sided'   # Hipótesis bilateral: cualquier diferencia
            )

            # Estadístico de efecto: rank-biserial correlation
            # r = 1 - (2U / n1*n2), rango [-1, 1]
            n1, n2 = len(datos_c1), len(datos_c2)
            r_efecto = 1 - (2 * stat) / (n1 * n2)

        else:
            stat, pval, r_efecto = np.nan, np.nan, np.nan

        resultados.append({
            'Variable'       : var,
            'Centro_1'       : c1,
            'Centro_2'       : c2,
            'Par'            : f"{c1} vs {c2}",
            'n_C1'           : len(datos_c1),
            'n_C2'           : len(datos_c2),
            'U_statistic'    : round(stat, 2) if not np.isnan(stat) else np.nan,
            'p_valor_raw'    : round(pval, 6) if not np.isnan(pval) else np.nan,
            'Efecto_r'       : round(r_efecto, 4) if not np.isnan(r_efecto) else np.nan
        })

df_resultados = pd.DataFrame(resultados)

# ==========================================
# 3. Corrección Benjamini-Hochberg
# ==========================================
mask_validos = df_resultados['p_valor_raw'].notna()

_, p_corregidos, _, _ = multipletests(
    df_resultados.loc[mask_validos, 'p_valor_raw'],
    alpha=0.05,
    method='fdr_bh'
)

df_resultados.loc[mask_validos, 'p_valor_BH']  = p_corregidos.round(6)
df_resultados['Significativo_BH'] = df_resultados['p_valor_BH'] < 0.05

# Interpretación del tamaño de efecto (Cohen 1988 adaptado para r)
def interpretar_efecto(r):
    r = abs(r)
    if r < 0.1:  return 'Despreciable'
    elif r < 0.3: return 'Pequeño'
    elif r < 0.5: return 'Moderado'
    else:         return 'Grande'

df_resultados['Magnitud_Efecto'] = df_resultados['Efecto_r'].apply(
    lambda x: interpretar_efecto(x) if not np.isnan(x) else np.nan
)

# ==========================================
# 4. Exportar tabla
# ==========================================
ruta_tabla = os.path.join(carpeta_stats, "MannWhitney_Comparacion_Centros.csv")
df_resultados.to_csv(ruta_tabla, index=False)
print(f"Tabla exportada: {ruta_tabla}")

# Resumen en consola
print(f"\nTests significativos (p_BH < 0.05): "
      f"{df_resultados['Significativo_BH'].sum()} de {mask_validos.sum()} comparaciones\n")

print(df_resultados[[
    'Variable', 'Par', 'p_valor_raw', 'p_valor_BH', 'Significativo_BH', 'Magnitud_Efecto'
]].to_string(index=False))

# ==========================================
# 5. Gráfico: Heatmap de p-values corregidos
# ==========================================
# Pivotamos para tener Variables en filas y Pares en columnas
df_pivot_pval = df_resultados.pivot(
    index='Variable', columns='Par', values='p_valor_BH'
)
df_pivot_sig = df_resultados.pivot(
    index='Variable', columns='Par', values='Significativo_BH'
)
df_pivot_efecto = df_resultados.pivot(
    index='Variable', columns='Par', values='Efecto_r'
)

# Renombrar índice con etiquetas legibles
df_pivot_pval.index   = [etiquetas_variables[v] for v in df_pivot_pval.index]
df_pivot_sig.index    = [etiquetas_variables[v] for v in df_pivot_sig.index]
df_pivot_efecto.index = [etiquetas_variables[v] for v in df_pivot_efecto.index]

sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# --- Panel izquierdo: p-values BH ---
ax1 = axes[0]
cmap_pval = sns.color_palette("RdYlGn_r", as_cmap=True)

sns.heatmap(
    df_pivot_pval,
    ax=ax1,
    cmap=cmap_pval,
    vmin=0, vmax=0.1,
    annot=True, fmt=".4f",
    linewidths=0.5,
    cbar_kws={"label": "p-valor (BH)", "shrink": 0.8},
    annot_kws={"size": 10}
)

# Marcar con asterisco las celdas significativas
for i in range(df_pivot_sig.shape[0]):
    for j in range(df_pivot_sig.shape[1]):
        if df_pivot_sig.iloc[i, j]:
            ax1.text(
                j + 0.85, i + 0.2, '*',
                ha='center', va='center',
                fontsize=14, color='black', fontweight='bold'
            )

ax1.set_title('p-valor corregido (Benjamini-Hochberg)\n* p < 0.05', fontsize=12, fontweight='bold')
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.tick_params(axis='x', rotation=15)
ax1.tick_params(axis='y', rotation=0)

# --- Panel derecho: tamaño de efecto (r rank-biserial) ---
ax2 = axes[1]
cmap_efecto = sns.diverging_palette(220, 20, as_cmap=True)

sns.heatmap(
    df_pivot_efecto,
    ax=ax2,
    cmap=cmap_efecto,
    vmin=-1, vmax=1, center=0,
    annot=True, fmt=".3f",
    linewidths=0.5,
    cbar_kws={"label": "r (rank-biserial)", "shrink": 0.8},
    annot_kws={"size": 10}
)

ax2.set_title('Tamaño de Efecto\n(correlación rank-biserial)', fontsize=12, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.tick_params(axis='x', rotation=15)
ax2.tick_params(axis='y', rotation=0)

plt.suptitle(
    'Comparación de Variables Oceanográficas entre Centros — Mann-Whitney U',
    fontsize=14, fontweight='bold', y=1.03
)

plt.tight_layout()
ruta_grafico = os.path.join(carpeta_stats, "MannWhitney_Comparacion_Centros.png")
plt.savefig(ruta_grafico, bbox_inches='tight')
plt.close()
print(f"\nGráfico exportado: {ruta_grafico}")
print("\nPunto 4 completado.")

In [ ]:
"""
ANÁLISIS TEMPORAL - Comparación entre Centros a lo largo del tiempo
====================================================================
Complementa el Mann-Whitney global (MannWhitney_Comparacion_Centros.csv/.png)
con una dimensión temporal. Dos análisis:

  A) Serie de tiempo comparativa: las 5 variables físicas con los 3 centros
     superpuestos en el mismo eje temporal.

  B) Mann-Whitney por año y por estación entre pares de centros.
     El p-value se reporta en notación científica para ver el valor real.

Outputs en Resultados_Exploracion/Estadisticas/:
  - SeriesTiempo_Comparativa_Centros.png
  - MannWhitney_Temporal_Anual.csv
  - MannWhitney_Temporal_Estacional.csv
  - MannWhitney_Temporal_Anual.png
  - MannWhitney_Temporal_Estacional.png

NO sobrescribe los archivos del análisis global previo.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import os

# ==========================================
# 1. Configuración
# ==========================================
ruta_dataset  = "data/Dataset_Maestro_Global_ML.csv"
carpeta_stats = "Resultados_Exploracion/Estadisticas/"
os.makedirs(carpeta_stats, exist_ok=True)

df = pd.read_csv(ruta_dataset)
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Año']   = df['Fecha'].dt.year

df['Velocidad_Corriente_m_s'] = np.sqrt(
    df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2
)

# Asignación estacional hemisferio sur
def asignar_estacion(fecha):
    mes = fecha.month
    if mes in [12, 1, 2]:   return 'Verano'
    elif mes in [3, 4, 5]:  return 'Otoño'
    elif mes in [6, 7, 8]:  return 'Invierno'
    else:                   return 'Primavera'

df['Estacion'] = df['Fecha'].apply(asignar_estacion)

centros_objetivo = [110557, 120069, 110745]
colores_centros  = {
    110557: '#1f77b4',
    120069: '#d62728',
    110745: '#2ca02c'
}
nombres_centros  = {
    110557: "Centro 110557",
    120069: "Centro 120069",
    110745: "Centro 110745"
}

variables_fisicas    = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m',
                        'Altura_Mar_m', 'Velocidad_Corriente_m_s']
etiquetas_variables  = {
    'Temperatura_C'          : 'Temperatura (°C)',
    'Salinidad_psu'          : 'Salinidad (psu)',
    'Capa_Mezcla_m'          : 'Capa de Mezcla (m)',
    'Altura_Mar_m'           : 'Altura del Mar (m)',
    'Velocidad_Corriente_m_s': 'Vel. Corriente (m/s)'
}

pares = list(combinations(centros_objetivo, 2))

# ==========================================
# A) Serie de Tiempo Comparativa
# ==========================================
print("Generando series de tiempo comparativas...")

# Media semanal por centro para suavizar la visualización
df_semanal = (
    df.groupby(['id_centro', pd.Grouper(key='Fecha', freq='W')])[variables_fisicas]
    .mean()
    .reset_index()
)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(len(variables_fisicas), 1,
                         figsize=(18, 4 * len(variables_fisicas)),
                         dpi=150, sharex=True)

for idx, var in enumerate(variables_fisicas):
    ax = axes[idx]
    for centro in centros_objetivo:
        datos = df_semanal[df_semanal['id_centro'] == centro].sort_values('Fecha')
        ax.plot(
            datos['Fecha'], datos[var],
            color=colores_centros[centro],
            label=nombres_centros[centro],
            linewidth=1.4, alpha=0.85
        )

    ax.set_ylabel(etiquetas_variables[var], fontsize=10)
    ax.yaxis.set_label_position('left')

    # Sombreado por año para facilitar lectura temporal
    años = df_semanal['Fecha'].dt.year.unique()
    for año in años:
        inicio = pd.Timestamp(f'{año}-01-01')
        fin    = pd.Timestamp(f'{año}-12-31')
        if año % 2 == 0:
            ax.axvspan(inicio, fin, alpha=0.04, color='grey')

    if idx == 0:
        ax.legend(loc='upper right', fontsize=9, framealpha=0.7)

axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right')
axes[-1].set_xlabel('Fecha', fontsize=11)

plt.suptitle(
    'Series de Tiempo de Variables Oceanográficas por Centro (media semanal)',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
ruta_series = os.path.join(carpeta_stats, "SeriesTiempo_Comparativa_Centros.png")
plt.savefig(ruta_series, bbox_inches='tight')
plt.close()
print(f"  Exportado: {ruta_series}")


# ==========================================
# B.1) Mann-Whitney por AÑO
# ==========================================
print("\nCalculando Mann-Whitney por año...")

resultados_anual = []
años_disponibles = sorted(df['Año'].unique())

for año in años_disponibles:
    df_año = df[df['Año'] == año]
    for var in variables_fisicas:
        for (c1, c2) in pares:
            d1 = df_año[df_año['id_centro'] == c1][var].dropna().values
            d2 = df_año[df_año['id_centro'] == c2][var].dropna().values

            if len(d1) > 5 and len(d2) > 5:
                stat, pval = stats.mannwhitneyu(d1, d2, alternative='two-sided')
                n1, n2     = len(d1), len(d2)
                r_efecto   = 1 - (2 * stat) / (n1 * n2)
            else:
                stat, pval, r_efecto = np.nan, np.nan, np.nan

            resultados_anual.append({
                'Año'         : año,
                'Variable'    : var,
                'Par'         : f"{c1} vs {c2}",
                'n_C1'        : len(d1),
                'n_C2'        : len(d2),
                'U_statistic' : round(stat, 2) if not np.isnan(stat) else np.nan,
                'p_valor_raw' : pval,     # Sin redondear aún — se redondea después
                'Efecto_r'    : round(r_efecto, 4) if not np.isnan(r_efecto) else np.nan
            })

df_anual = pd.DataFrame(resultados_anual)

# Corrección BH sobre todos los tests anuales juntos
mask = df_anual['p_valor_raw'].notna()
_, p_corr, _, _ = multipletests(df_anual.loc[mask, 'p_valor_raw'], alpha=0.05, method='fdr_bh')
df_anual.loc[mask, 'p_valor_BH'] = p_corr
df_anual['Significativo_BH']     = df_anual['p_valor_BH'] < 0.05

# Notación científica para el reporte
df_anual['p_valor_raw_str'] = df_anual['p_valor_raw'].apply(
    lambda x: f"{x:.4e}" if not np.isnan(x) else "NA"
)
df_anual['p_valor_BH_str'] = df_anual['p_valor_BH'].apply(
    lambda x: f"{x:.4e}" if not np.isnan(x) else "NA"
)

ruta_anual = os.path.join(carpeta_stats, "MannWhitney_Temporal_Anual.csv")
df_anual.to_csv(ruta_anual, index=False)
print(f"  Exportado: {ruta_anual}")


# ==========================================
# B.2) Mann-Whitney por ESTACIÓN
# ==========================================
print("\nCalculando Mann-Whitney por estación...")

orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
resultados_estacional = []

for estacion in orden_estaciones:
    df_est = df[df['Estacion'] == estacion]
    for var in variables_fisicas:
        for (c1, c2) in pares:
            d1 = df_est[df_est['id_centro'] == c1][var].dropna().values
            d2 = df_est[df_est['id_centro'] == c2][var].dropna().values

            if len(d1) > 5 and len(d2) > 5:
                stat, pval = stats.mannwhitneyu(d1, d2, alternative='two-sided')
                n1, n2     = len(d1), len(d2)
                r_efecto   = 1 - (2 * stat) / (n1 * n2)
            else:
                stat, pval, r_efecto = np.nan, np.nan, np.nan

            resultados_estacional.append({
                'Estacion'    : estacion,
                'Variable'    : var,
                'Par'         : f"{c1} vs {c2}",
                'n_C1'        : len(d1),
                'n_C2'        : len(d2),
                'U_statistic' : round(stat, 2) if not np.isnan(stat) else np.nan,
                'p_valor_raw' : pval,
                'Efecto_r'    : round(r_efecto, 4) if not np.isnan(r_efecto) else np.nan
            })

df_estacional = pd.DataFrame(resultados_estacional)

mask_e = df_estacional['p_valor_raw'].notna()
_, p_corr_e, _, _ = multipletests(
    df_estacional.loc[mask_e, 'p_valor_raw'], alpha=0.05, method='fdr_bh'
)
df_estacional.loc[mask_e, 'p_valor_BH'] = p_corr_e
df_estacional['Significativo_BH']        = df_estacional['p_valor_BH'] < 0.05

df_estacional['p_valor_raw_str'] = df_estacional['p_valor_raw'].apply(
    lambda x: f"{x:.4e}" if not np.isnan(x) else "NA"
)
df_estacional['p_valor_BH_str'] = df_estacional['p_valor_BH'].apply(
    lambda x: f"{x:.4e}" if not np.isnan(x) else "NA"
)

ruta_estacional = os.path.join(carpeta_stats, "MannWhitney_Temporal_Estacional.csv")
df_estacional.to_csv(ruta_estacional, index=False)
print(f"  Exportado: {ruta_estacional}")


# ==========================================
# C) Gráficos temporales del tamaño de efecto
# ==========================================

# --- C.1) Heatmap tamaño de efecto por AÑO ---
print("\nGenerando gráficos temporales...")

# Un panel por par de centros
fig, axes = plt.subplots(1, len(pares), figsize=(7 * len(pares), 5), dpi=150)
cmap_div = sns.diverging_palette(220, 20, as_cmap=True)

for idx, par in enumerate([f"{c1} vs {c2}" for c1, c2 in pares]):
    df_par = df_anual[df_anual['Par'] == par].copy()
    df_par['Variable_label'] = df_par['Variable'].map(etiquetas_variables)

    pivot = df_par.pivot(index='Variable_label', columns='Año', values='Efecto_r')
    sig   = df_par.pivot(index='Variable_label', columns='Año', values='Significativo_BH')

    ax = axes[idx]
    sns.heatmap(
        pivot, ax=ax,
        cmap=cmap_div, vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f", linewidths=0.4,
        annot_kws={"size": 8},
        cbar=(idx == len(pares) - 1),
        cbar_kws={"label": "r (rank-biserial)", "shrink": 0.8}
    )
    # Asterisco en celdas significativas
    for i in range(sig.shape[0]):
        for j in range(sig.shape[1]):
            if pd.notna(sig.iloc[i, j]) and sig.iloc[i, j]:
                ax.text(j + 0.82, i + 0.22, '*',
                        ha='center', va='center',
                        fontsize=12, color='black', fontweight='bold')

    ax.set_title(f'Par: {par}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Año')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle(
    'Tamaño de Efecto (Mann-Whitney) por Año — Comparación entre Centros\n* p < 0.05 (BH)',
    fontsize=13, fontweight='bold', y=1.03
)
plt.tight_layout()
ruta_graf_anual = os.path.join(carpeta_stats, "MannWhitney_Temporal_Anual.png")
plt.savefig(ruta_graf_anual, bbox_inches='tight')
plt.close()
print(f"  Exportado: {ruta_graf_anual}")


# --- C.2) Heatmap tamaño de efecto por ESTACIÓN ---
fig, axes = plt.subplots(1, len(pares), figsize=(7 * len(pares), 5), dpi=150)

for idx, par in enumerate([f"{c1} vs {c2}" for c1, c2 in pares]):
    df_par = df_estacional[df_estacional['Par'] == par].copy()
    df_par['Variable_label'] = df_par['Variable'].map(etiquetas_variables)

    pivot = df_par.pivot(index='Variable_label', columns='Estacion', values='Efecto_r')
    sig   = df_par.pivot(index='Variable_label', columns='Estacion', values='Significativo_BH')

    # Reordenar columnas según orden estacional
    pivot = pivot[[e for e in orden_estaciones if e in pivot.columns]]
    sig   = sig[[e for e in orden_estaciones if e in sig.columns]]

    ax = axes[idx]
    sns.heatmap(
        pivot, ax=ax,
        cmap=cmap_div, vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f", linewidths=0.4,
        annot_kws={"size": 9},
        cbar=(idx == len(pares) - 1),
        cbar_kws={"label": "r (rank-biserial)", "shrink": 0.8}
    )
    for i in range(sig.shape[0]):
        for j in range(sig.shape[1]):
            if pd.notna(sig.iloc[i, j]) and sig.iloc[i, j]:
                ax.text(j + 0.82, i + 0.22, '*',
                        ha='center', va='center',
                        fontsize=12, color='black', fontweight='bold')

    ax.set_title(f'Par: {par}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Estación')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle(
    'Tamaño de Efecto (Mann-Whitney) por Estación — Comparación entre Centros\n* p < 0.05 (BH)',
    fontsize=13, fontweight='bold', y=1.03
)
plt.tight_layout()
ruta_graf_est = os.path.join(carpeta_stats, "MannWhitney_Temporal_Estacional.png")
plt.savefig(ruta_graf_est, bbox_inches='tight')
plt.close()
print(f"  Exportado: {ruta_graf_est}")

print("\nAnálisis temporal completado.")
print("\nArchivos generados:")
print(f"  {ruta_series}")
print(f"  {ruta_anual}")
print(f"  {ruta_estacional}")
print(f"  {ruta_graf_anual}")
print(f"  {ruta_graf_est}")

In [ ]:
"""
DIAGNÓSTICO DE NORMALIDAD
=========================
Flujo metodológico previo a la elección de pruebas estadísticas:

  1. Kolmogorov-Smirnov con corrección de Lilliefors
     (apropiado para n grande, a diferencia de Shapiro-Wilk que pierde validez con n > ~2000)
  2. Q-Q plots por variable y centro (evidencia visual)
  3. Histogramas con curva KDE superpuesta (evidencia visual complementaria)

Con estos resultados se fundamenta formalmente si Mann-Whitney U
fue la elección correcta frente a t de Student o t de Welch.

Outputs en Resultados_Exploracion/Estadisticas/:
  - Normalidad_Lilliefors.csv     → tabla con estadístico D, p-value y decisión
  - Normalidad_QQ_Plots.png       → Q-Q plots por variable y centro
  - Normalidad_Histogramas.png    → histogramas con KDE por variable y centro
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats as stats
from statsmodels.stats.diagnostic import lilliefors
import os

# ==========================================
# 1. Configuración
# ==========================================
ruta_dataset  = "data/Dataset_Maestro_Global_ML.csv"
carpeta_stats = "Resultados_Exploracion/Estadisticas/"
os.makedirs(carpeta_stats, exist_ok=True)

df = pd.read_csv(ruta_dataset)

df['Velocidad_Corriente_m_s'] = np.sqrt(
    df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2
)

centros_objetivo  = [110557, 120069, 110745]
nombres_centros   = {
    110557: "Centro 110557",
    120069: "Centro 120069",
    110745: "Centro 110745"
}
colores_centros   = {
    110557: '#1f77b4',
    120069: '#d62728',
    110745: '#2ca02c'
}

variables_fisicas = [
    'Temperatura_C',
    'Salinidad_psu',
    'Capa_Mezcla_m',
    'Altura_Mar_m',
    'Velocidad_Corriente_m_s'
]
variables_biologicas = [
    'Juveniles',
    'Adultos_Moviles',
    'Hembras_Ovigeras'
]

# Para biológicas: excluir periodos de descanso (registros con NaN en parasitos)
# Para oceanográficas: usar todos los registros disponibles
todas_variables = variables_fisicas + variables_biologicas

etiquetas = {
    'Temperatura_C'          : 'Temperatura (°C)',
    'Salinidad_psu'          : 'Salinidad (psu)',
    'Capa_Mezcla_m'          : 'Capa de Mezcla (m)',
    'Altura_Mar_m'           : 'Altura del Mar (m)',
    'Velocidad_Corriente_m_s': 'Vel. Corriente (m/s)',
    'Juveniles'              : 'Juveniles',
    'Adultos_Moviles'        : 'Adultos Móviles',
    'Hembras_Ovigeras'       : 'Hembras Ovígeras'
}

print("Iniciando diagnóstico de normalidad (Lilliefors + gráficos)...\n")

# ==========================================
# 2. Test de Lilliefors por variable y centro
# ==========================================
resultados_normalidad = []

for centro in centros_objetivo:
    df_centro_ocean = df[df['id_centro'] == centro]

    # Para biológicas filtramos NaN (periodos de descanso)
    df_centro_bio = df_centro_ocean.dropna(subset=variables_biologicas)

    for var in todas_variables:

        # Seleccionar el dataframe correcto según tipo de variable
        if var in variables_biologicas:
            datos = df_centro_bio[var].dropna().values
        else:
            datos = df_centro_ocean[var].dropna().values

        n = len(datos)

        if n > 10:
            # Lilliefors: KS corregido para parámetros estimados desde los datos
            # H0: los datos siguen una distribución normal
            # Si p < 0.05 → rechazamos normalidad
            stat_D, pval = lilliefors(datos, dist='norm', pvalmethod='approx')

            # Asimetría y curtosis como métricas adicionales
            asimetria = stats.skew(datos)
            curtosis  = stats.kurtosis(datos)  # Exceso de curtosis (normal = 0)
        else:
            stat_D, pval, asimetria, curtosis = np.nan, np.nan, np.nan, np.nan

        resultados_normalidad.append({
            'Centro'         : centro,
            'Variable'       : var,
            'Tipo'           : 'Oceanográfica' if var in variables_fisicas else 'Biológica',
            'n'              : n,
            'D_Lilliefors'   : round(stat_D, 6) if not np.isnan(stat_D) else np.nan,
            'p_valor'        : f"{pval:.4e}" if not np.isnan(pval) else "NA",
            'Rechaza_H0'     : pval < 0.05 if not np.isnan(pval) else np.nan,
            'Asimetria'      : round(asimetria, 4) if not np.isnan(asimetria) else np.nan,
            'Curtosis_exceso': round(curtosis, 4) if not np.isnan(curtosis) else np.nan,
            'Conclusion'     : 'No normal' if (not np.isnan(pval) and pval < 0.05) else 'Normal'
        })

df_normalidad = pd.DataFrame(resultados_normalidad)

# Resumen en consola
total      = df_normalidad['Rechaza_H0'].notna().sum()
no_normal  = df_normalidad['Rechaza_H0'].sum()
print(f"Resultados Lilliefors:")
print(f"  Variables no normales (p < 0.05): {int(no_normal)} de {total}")
print(f"  Variables normales:               {int(total - no_normal)} de {total}\n")

# Detalle de las que podrían ser normales
normales = df_normalidad[df_normalidad['Rechaza_H0'] == False]
if len(normales) > 0:
    print("Variables que NO rechazan normalidad:")
    print(normales[['Centro', 'Variable', 'n', 'p_valor', 'Asimetria']].to_string(index=False))
else:
    print("Todas las variables rechazan normalidad → Mann-Whitney U fue la elección correcta.\n")

# Exportar tabla
ruta_tabla = os.path.join(carpeta_stats, "Normalidad_Lilliefors.csv")
df_normalidad.to_csv(ruta_tabla, index=False)
print(f"\nTabla exportada: {ruta_tabla}")


# ==========================================
# 3. Q-Q Plots
# ==========================================
# Un panel por variable, tres subpaneles (uno por centro)
print("\nGenerando Q-Q plots...")

n_vars = len(todas_variables)
fig, axes = plt.subplots(n_vars, 3, figsize=(15, 4 * n_vars), dpi=130)

for i, var in enumerate(todas_variables):
    for j, centro in enumerate(centros_objetivo):
        ax = axes[i, j]

        df_centro = df[df['id_centro'] == centro]
        if var in variables_biologicas:
            datos = df_centro.dropna(subset=variables_biologicas)[var].dropna().values
        else:
            datos = df_centro[var].dropna().values

        if len(datos) > 10:
            # Q-Q plot: si los puntos siguen la línea roja → distribución normal
            (osm, osr), (slope, intercept, r) = stats.probplot(datos, dist='norm')
            ax.scatter(osm, osr, color=colores_centros[centro],
                       alpha=0.3, s=4, rasterized=True)
            ax.plot(
                [min(osm), max(osm)],
                [slope * min(osm) + intercept, slope * max(osm) + intercept],
                color='black', linewidth=1.5, linestyle='--', label=f'R²={r**2:.3f}'
            )
            ax.legend(fontsize=8, loc='upper left')
        else:
            ax.text(0.5, 0.5, 'n insuficiente', ha='center', va='center',
                    transform=ax.transAxes, color='grey')

        if i == 0:
            ax.set_title(nombres_centros[centro], fontsize=11, fontweight='bold')
        if j == 0:
            ax.set_ylabel(etiquetas[var], fontsize=9)
        if i == n_vars - 1:
            ax.set_xlabel('Cuantiles teóricos\n(distribución normal)', fontsize=8)

        ax.tick_params(labelsize=7)

plt.suptitle(
    'Diagnóstico de Normalidad — Q-Q Plots por Variable y Centro\n'
    '(desviación de la línea diagonal indica no normalidad)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
ruta_qq = os.path.join(carpeta_stats, "Normalidad_QQ_Plots.png")
plt.savefig(ruta_qq, bbox_inches='tight')
plt.close()
print(f"  Exportado: {ruta_qq}")


# ==========================================
# 4. Histogramas con KDE
# ==========================================
print("Generando histogramas con KDE...")

fig, axes = plt.subplots(n_vars, 3, figsize=(15, 4 * n_vars), dpi=130)

for i, var in enumerate(todas_variables):
    for j, centro in enumerate(centros_objetivo):
        ax = axes[i, j]
        color = colores_centros[centro]

        df_centro = df[df['id_centro'] == centro]
        if var in variables_biologicas:
            datos = df_centro.dropna(subset=variables_biologicas)[var].dropna().values
        else:
            datos = df_centro[var].dropna().values

        if len(datos) > 10:
            # Histograma normalizado (densidad)
            ax.hist(datos, bins=40, color=color, alpha=0.45,
                    density=True, edgecolor='white', linewidth=0.3)

            # KDE empírica
            kde = stats.gaussian_kde(datos)
            x_range = np.linspace(datos.min(), datos.max(), 300)
            ax.plot(x_range, kde(x_range), color=color, linewidth=2)

            # Curva normal teórica superpuesta (misma media y std)
            mu, sigma = datos.mean(), datos.std()
            ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma),
                    color='black', linewidth=1.5, linestyle='--',
                    label='Normal teórica')

            # Asimetría en el gráfico
            skew_val = stats.skew(datos)
            ax.text(0.97, 0.95, f'Asim: {skew_val:.2f}',
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=8, color='black',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

            if i == 0 and j == 0:
                ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'n insuficiente', ha='center', va='center',
                    transform=ax.transAxes, color='grey')

        if i == 0:
            ax.set_title(nombres_centros[centro], fontsize=11, fontweight='bold')
        if j == 0:
            ax.set_ylabel(etiquetas[var], fontsize=9)
        if i == n_vars - 1:
            ax.set_xlabel('Valor', fontsize=8)

        ax.tick_params(labelsize=7)

plt.suptitle(
    'Diagnóstico de Normalidad — Histogramas con KDE por Variable y Centro\n'
    '(línea discontinua: distribución normal teórica equivalente)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
ruta_hist = os.path.join(carpeta_stats, "Normalidad_Histogramas.png")
plt.savefig(ruta_hist, bbox_inches='tight')
plt.close()
print(f"  Exportado: {ruta_hist}")

print("\nDiagnóstico de normalidad completado.")
print("\nArchivos generados:")
print(f"  {ruta_tabla}")
print(f"  {ruta_qq}")
print(f"  {ruta_hist}")
print("\nConclusión metodológica:")
print(f"  Si todas o la mayoría rechazan normalidad → Mann-Whitney U fue correcto.")
print(f"  Si alguna no rechaza → para esas variables específicas se podría")
print(f"  haber usado t de Welch, pero Mann-Whitney sigue siendo válido.")

In [ ]:
"""
PUNTO 5 - Diferencias Estacionales: Kruskal-Wallis + Dunn post-hoc
====================================================================
Dentro de cada centro, compara las cuatro estaciones (Verano, Otoño,
Invierno, Primavera) para cada variable oceanográfica y biológica.

Flujo:
  1. Kruskal-Wallis → ¿hay alguna diferencia entre estaciones? (test omnibus)
  2. Si p < 0.05 → Dunn post-hoc → ¿qué par de estaciones es distinto?
  3. Corrección Benjamini-Hochberg sobre todos los p-values de Dunn

Variables oceanográficas → todos los registros disponibles
Variables biológicas     → excluir periodos de descanso (NaN parasitarios)

Outputs en Resultados_Exploracion/Estadisticas/:
  - Kruskal_Resultados.csv          → tabla Kruskal-Wallis por variable y centro
  - Dunn_Resultados.csv             → tabla post-hoc Dunn por par de estaciones
  - Kruskal_Heatmap.png             → significancia y efecto por variable/centro
  - Dunn_Heatmap_CentroXXX.png      → un heatmap por centro con pares estacionales
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import os

# ==========================================
# 1. Configuración
# ==========================================
ruta_dataset  = "data/Dataset_Maestro_Global_ML.csv"
carpeta_stats = "Resultados_Exploracion/Estadisticas/"
os.makedirs(carpeta_stats, exist_ok=True)

df = pd.read_csv(ruta_dataset)
df['Fecha'] = pd.to_datetime(df['Fecha'])

df['Velocidad_Corriente_m_s'] = np.sqrt(
    df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2
)

# Asignación estacional hemisferio sur
def asignar_estacion(fecha):
    mes = fecha.month
    if mes in [12, 1, 2]:  return 'Verano'
    elif mes in [3, 4, 5]: return 'Otoño'
    elif mes in [6, 7, 8]: return 'Invierno'
    else:                  return 'Primavera'

df['Estacion'] = df['Fecha'].apply(asignar_estacion)

centros_objetivo  = [110557, 120069, 110745]
nombres_centros   = {
    110557: "Centro 110557",
    120069: "Centro 120069",
    110745: "Centro 110745"
}

orden_estaciones  = ['Verano', 'Otoño', 'Invierno', 'Primavera']
pares_estaciones  = list(combinations(orden_estaciones, 2))  # 6 pares

variables_fisicas    = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m',
                        'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_biologicas = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
todas_variables      = variables_fisicas + variables_biologicas

etiquetas = {
    'Temperatura_C'          : 'Temperatura (°C)',
    'Salinidad_psu'          : 'Salinidad (psu)',
    'Capa_Mezcla_m'          : 'Capa de Mezcla (m)',
    'Altura_Mar_m'           : 'Altura del Mar (m)',
    'Velocidad_Corriente_m_s': 'Vel. Corriente (m/s)',
    'Juveniles'              : 'Juveniles',
    'Adultos_Moviles'        : 'Adultos Móviles',
    'Hembras_Ovigeras'       : 'Hembras Ovígeras'
}

print("Iniciando Kruskal-Wallis + Dunn post-hoc por estación...\n")

# ==========================================
# 2. Kruskal-Wallis por variable y centro
# ==========================================
resultados_kruskal = []

for centro in centros_objetivo:
    df_centro_ocean = df[df['id_centro'] == centro]
    df_centro_bio   = df_centro_ocean.dropna(subset=variables_biologicas)

    for var in todas_variables:
        df_base = df_centro_bio if var in variables_biologicas else df_centro_ocean

        # Agrupar por estación
        grupos = [
            df_base[df_base['Estacion'] == est][var].dropna().values
            for est in orden_estaciones
        ]
        # Solo incluir estaciones con datos suficientes
        grupos_validos  = [g for g in grupos if len(g) > 5]
        nombres_validos = [e for e, g in zip(orden_estaciones, grupos) if len(g) > 5]

        if len(grupos_validos) >= 2:
            H, pval = stats.kruskal(*grupos_validos)

            # Eta cuadrado (η²) como tamaño de efecto para Kruskal-Wallis
            # η² = (H - k + 1) / (n - k), donde k = n° grupos, n = n total
            n_total = sum(len(g) for g in grupos_validos)
            k       = len(grupos_validos)
            eta2    = (H - k + 1) / (n_total - k)
            eta2    = max(0, eta2)  # No puede ser negativo

            def interpretar_eta2(e):
                if e < 0.01:   return 'Despreciable'
                elif e < 0.06: return 'Pequeño'
                elif e < 0.14: return 'Moderado'
                else:          return 'Grande'
        else:
            H, pval, eta2 = np.nan, np.nan, np.nan

        resultados_kruskal.append({
            'Centro'          : centro,
            'Variable'        : var,
            'Tipo'            : 'Biológica' if var in variables_biologicas else 'Oceanográfica',
            'n_total'         : sum(len(g) for g in grupos_validos),
            'H_statistic'     : round(H, 4) if not np.isnan(H) else np.nan,
            'p_valor'         : f"{pval:.4e}" if not np.isnan(pval) else "NA",
            'p_valor_num'     : pval,
            'Significativo'   : pval < 0.05 if not np.isnan(pval) else np.nan,
            'Eta2'            : round(eta2, 4) if not np.isnan(eta2) else np.nan,
            'Magnitud_Efecto' : interpretar_eta2(eta2) if not np.isnan(eta2) else np.nan
        })

df_kruskal = pd.DataFrame(resultados_kruskal)

# Resumen consola
sig = df_kruskal['Significativo'].sum()
tot = df_kruskal['Significativo'].notna().sum()
print(f"Kruskal-Wallis significativos (p < 0.05): {int(sig)} de {tot}\n")
print(df_kruskal[['Centro', 'Variable', 'p_valor',
                   'Significativo', 'Eta2', 'Magnitud_Efecto']].to_string(index=False))

# Exportar tabla Kruskal
ruta_kruskal = os.path.join(carpeta_stats, "Kruskal_Resultados.csv")
df_kruskal.drop(columns='p_valor_num').to_csv(ruta_kruskal, index=False)
print(f"\nTabla Kruskal exportada: {ruta_kruskal}")


# ==========================================
# 3. Dunn post-hoc donde Kruskal fue significativo
# ==========================================
# Implementación manual de Dunn sin dependencias adicionales.
# Dunn usa los rangos globales del dataset completo por grupo.

def dunn_test(grupos, nombres):
    """
    Test de Dunn pairwise para los grupos dados.
    Retorna DataFrame con par, Z, p_valor_raw.
    """
    # Combinar y rankear todos los datos juntos
    todos   = np.concatenate(grupos)
    rangos  = stats.rankdata(todos)

    # Reconstruir rangos por grupo
    idx = 0
    rangos_grupos = []
    for g in grupos:
        rangos_grupos.append(rangos[idx:idx + len(g)])
        idx += len(g)

    n_total = len(todos)
    resultados = []

    for (i, j) in combinations(range(len(grupos)), 2):
        ni, nj   = len(grupos[i]), len(grupos[j])
        Ri, Rj   = rangos_grupos[i].mean(), rangos_grupos[j].mean()

        # Corrección por empates
        _, counts = np.unique(todos, return_counts=True)
        T = np.sum(counts**3 - counts) / (12 * (n_total - 1))

        SE = np.sqrt(
            (n_total * (n_total + 1) / 12 - T) * (1/ni + 1/nj)
        )
        Z  = (Ri - Rj) / SE if SE > 0 else np.nan
        p  = 2 * stats.norm.sf(abs(Z)) if not np.isnan(Z) else np.nan

        resultados.append({
            'Estacion_1'  : nombres[i],
            'Estacion_2'  : nombres[j],
            'Par'         : f"{nombres[i]} vs {nombres[j]}",
            'Z'           : round(Z, 4) if not np.isnan(Z) else np.nan,
            'p_valor_raw' : p
        })

    return pd.DataFrame(resultados)


print("\nCalculando Dunn post-hoc...")
resultados_dunn = []

for centro in centros_objetivo:
    df_centro_ocean = df[df['id_centro'] == centro]
    df_centro_bio   = df_centro_ocean.dropna(subset=variables_biologicas)

    # Solo variables donde Kruskal fue significativo en este centro
    vars_sig = df_kruskal[
        (df_kruskal['Centro'] == centro) &
        (df_kruskal['Significativo'] == True)
    ]['Variable'].tolist()

    for var in vars_sig:
        df_base = df_centro_bio if var in variables_biologicas else df_centro_ocean

        grupos  = [df_base[df_base['Estacion'] == est][var].dropna().values
                   for est in orden_estaciones]
        nombres = [e for e, g in zip(orden_estaciones, grupos) if len(g) > 5]
        grupos  = [g for g in grupos if len(g) > 5]

        df_dunn_var = dunn_test(grupos, nombres)
        df_dunn_var['Centro']   = centro
        df_dunn_var['Variable'] = var
        resultados_dunn.append(df_dunn_var)

df_dunn_all = pd.concat(resultados_dunn, ignore_index=True)

# Corrección BH sobre TODOS los p-values de Dunn juntos
mask = df_dunn_all['p_valor_raw'].notna()
_, p_corr, _, _ = multipletests(
    df_dunn_all.loc[mask, 'p_valor_raw'], alpha=0.05, method='fdr_bh'
)
df_dunn_all.loc[mask, 'p_valor_BH']  = p_corr
df_dunn_all['Significativo_BH']      = df_dunn_all['p_valor_BH'] < 0.05
df_dunn_all['p_valor_raw_str']        = df_dunn_all['p_valor_raw'].apply(
    lambda x: f"{x:.4e}" if pd.notna(x) else "NA"
)
df_dunn_all['p_valor_BH_str']         = df_dunn_all['p_valor_BH'].apply(
    lambda x: f"{x:.4e}" if pd.notna(x) else "NA"
)

ruta_dunn = os.path.join(carpeta_stats, "Dunn_Resultados.csv")
df_dunn_all.drop(columns=['p_valor_raw']).to_csv(ruta_dunn, index=False)
print(f"Tabla Dunn exportada: {ruta_dunn}")


# ==========================================
# 4. Gráfico A — Heatmap Kruskal (η² por variable y centro)
# ==========================================
print("\nGenerando gráficos...")

df_pivot_eta = df_kruskal.pivot(index='Variable', columns='Centro', values='Eta2')
df_pivot_sig = df_kruskal.pivot(index='Variable', columns='Centro', values='Significativo')
df_pivot_eta.index = [etiquetas[v] for v in df_pivot_eta.index]
df_pivot_sig.index = [etiquetas[v] for v in df_pivot_sig.index]
df_pivot_eta.columns = [nombres_centros[c] for c in df_pivot_eta.columns]
df_pivot_sig.columns = [nombres_centros[c] for c in df_pivot_sig.columns]

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(9, 7), dpi=150)

sns.heatmap(
    df_pivot_eta, ax=ax,
    cmap='YlOrRd', vmin=0, vmax=0.3,
    annot=True, fmt=".3f", linewidths=0.5,
    cbar_kws={"label": "η² (eta cuadrado)", "shrink": 0.8},
    annot_kws={"size": 10}
)

# Asterisco donde es significativo
for i in range(df_pivot_sig.shape[0]):
    for j in range(df_pivot_sig.shape[1]):
        val = df_pivot_sig.iloc[i, j]
        if pd.notna(val) and val:
            ax.text(j + 0.82, i + 0.22, '*',
                    ha='center', va='center',
                    fontsize=13, color='black', fontweight='bold')

ax.set_title(
    'Kruskal-Wallis — Efecto Estacional por Variable y Centro\n* p < 0.05',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=15)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
ruta_kruskal_png = os.path.join(carpeta_stats, "Kruskal_Heatmap.png")
plt.savefig(ruta_kruskal_png, bbox_inches='tight')
plt.close()
print(f"  Exportado: {ruta_kruskal_png}")


# ==========================================
# 5. Gráfico B — Heatmap Dunn por centro
# ==========================================
for centro in centros_objetivo:
    df_centro_dunn = df_dunn_all[df_dunn_all['Centro'] == centro].copy()
    if df_centro_dunn.empty:
        continue

    df_centro_dunn['Variable_label'] = df_centro_dunn['Variable'].map(etiquetas)

    pivot_bh  = df_centro_dunn.pivot_table(
        index='Variable_label', columns='Par', values='p_valor_BH', aggfunc='first'
    )
    pivot_sig = df_centro_dunn.pivot_table(
        index='Variable_label', columns='Par', values='Significativo_BH', aggfunc='first'
    )

    # Ordenar columnas según orden estacional
    orden_pares = [f"{a} vs {b}" for a, b in pares_estaciones]
    cols_presentes = [c for c in orden_pares if c in pivot_bh.columns]
    pivot_bh  = pivot_bh[cols_presentes]
    pivot_sig = pivot_sig[cols_presentes]

    fig, ax = plt.subplots(
        figsize=(max(10, len(cols_presentes) * 1.8), max(5, len(pivot_bh) * 0.8 + 2)),
        dpi=150
    )

    cmap_pval = sns.color_palette("RdYlGn_r", as_cmap=True)
    sns.heatmap(
        pivot_bh, ax=ax,
        cmap=cmap_pval, vmin=0, vmax=0.1,
        annot=True, fmt=".4f", linewidths=0.5,
        cbar_kws={"label": "p-valor (BH)", "shrink": 0.8},
        annot_kws={"size": 9}
    )

    for i in range(pivot_sig.shape[0]):
        for j in range(pivot_sig.shape[1]):
            val = pivot_sig.iloc[i, j]
            if pd.notna(val) and val:
                ax.text(j + 0.82, i + 0.22, '*',
                        ha='center', va='center',
                        fontsize=12, color='black', fontweight='bold')

    ax.set_title(
        f'Dunn post-hoc — Comparaciones Estacionales\n{nombres_centros[centro]}  |  * p < 0.05 (BH)',
        fontsize=12, fontweight='bold', pad=12
    )
    ax.set_xlabel('Par de estaciones')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

    plt.tight_layout()
    ruta_dunn_png = os.path.join(carpeta_stats, f"Dunn_Heatmap_Centro{centro}.png")
    plt.savefig(ruta_dunn_png, bbox_inches='tight')
    plt.close()
    print(f"  Exportado: {ruta_dunn_png}")

print("\nPunto 5 completado.")
print("\nArchivos generados:")
print(f"  {ruta_kruskal}")
print(f"  {ruta_dunn}")
print(f"  {ruta_kruskal_png}")
for centro in centros_objetivo:
    print(f"  Resultados_Exploracion/Estadisticas/Dunn_Heatmap_Centro{centro}.png")

In [ ]:
"""
DIFERENCIAS ESTACIONALES — Estimador de Hodges-Lehmann
=======================================================
Calcula la diferencia numérica exacta entre pares de estaciones
para cada variable y centro, en las unidades originales de cada variable.

Método: Hodges-Lehmann
  - Toma todos los pares posibles de valores entre dos grupos (x_i - y_j)
  - La mediana de esas diferencias es el estimador puntual
  - El IC 95% se obtiene por bootstrap (2000 iteraciones)
  - Es el complemento no paramétrico natural de Mann-Whitney:
    donde Mann-Whitney dice "son distintos", Hodges-Lehmann dice "en cuánto"

Variables oceanográficas → todos los registros
Variables biológicas     → excluir periodos de descanso

Solo se calcula para los pares que resultaron significativos en Dunn.

Output en Resultados_Exploracion/Estadisticas/:
  - HodgesLehmann_Diferencias_Estacionales.csv
"""

import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import os

# ==========================================
# 1. Configuración
# ==========================================
ruta_dataset  = "data/Dataset_Maestro_Global_ML.csv"
carpeta_stats = "Resultados_Exploracion/Estadisticas/"
os.makedirs(carpeta_stats, exist_ok=True)

df = pd.read_csv(ruta_dataset)
df['Fecha'] = pd.to_datetime(df['Fecha'])

df['Velocidad_Corriente_m_s'] = np.sqrt(
    df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2
)

def asignar_estacion(fecha):
    mes = fecha.month
    if mes in [12, 1, 2]:  return 'Verano'
    elif mes in [3, 4, 5]: return 'Otoño'
    elif mes in [6, 7, 8]: return 'Invierno'
    else:                  return 'Primavera'

df['Estacion'] = df['Fecha'].apply(asignar_estacion)

centros_objetivo     = [110557, 120069, 110745]
orden_estaciones     = ['Verano', 'Otoño', 'Invierno', 'Primavera']
variables_fisicas    = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m',
                        'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_biologicas = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
todas_variables      = variables_fisicas + variables_biologicas

unidades = {
    'Temperatura_C'          : '°C',
    'Salinidad_psu'          : 'psu',
    'Capa_Mezcla_m'          : 'm',
    'Altura_Mar_m'           : 'm',
    'Velocidad_Corriente_m_s': 'm/s',
    'Juveniles'              : 'ind.',
    'Adultos_Moviles'        : 'ind.',
    'Hembras_Ovigeras'       : 'ind.'
}

etiquetas = {
    'Temperatura_C'          : 'Temperatura (°C)',
    'Salinidad_psu'          : 'Salinidad (psu)',
    'Capa_Mezcla_m'          : 'Capa de Mezcla (m)',
    'Altura_Mar_m'           : 'Altura del Mar (m)',
    'Velocidad_Corriente_m_s': 'Vel. Corriente (m/s)',
    'Juveniles'              : 'Juveniles',
    'Adultos_Moviles'        : 'Adultos Móviles',
    'Hembras_Ovigeras'       : 'Hembras Ovígeras'
}

N_BOOTSTRAP = 2000
np.random.seed(42)

# ==========================================
# 2. Cargar pares significativos de Dunn
# ==========================================
ruta_dunn = os.path.join(carpeta_stats, "Dunn_Resultados.csv")
df_dunn   = pd.read_csv(ruta_dunn)
df_sig    = df_dunn[df_dunn['Significativo_BH'] == True][
    ['Centro', 'Variable', 'Estacion_1', 'Estacion_2']
].drop_duplicates()

print(f"Pares significativos en Dunn: {len(df_sig)}")
print("Calculando Hodges-Lehmann con bootstrap IC 95%...\n")

# ==========================================
# 3. Estimador Hodges-Lehmann con bootstrap
# ==========================================
def hodges_lehmann(x, y):
    """
    Estimador puntual de Hodges-Lehmann: mediana de todas las diferencias (x_i - y_j).
    Interpretación: cuánto tiende a ser mayor x respecto a y.
    Positivo → x > y en mediana. Negativo → x < y en mediana.
    """
    diffs = np.subtract.outer(x, y).ravel()
    return np.median(diffs)

def bootstrap_ic(x, y, n_iter=2000, alpha=0.05):
    """
    IC bootstrap percentil para la diferencia de Hodges-Lehmann.
    """
    estimates = []
    for _ in range(n_iter):
        x_boot = np.random.choice(x, size=len(x), replace=True)
        y_boot = np.random.choice(y, size=len(y), replace=True)
        estimates.append(hodges_lehmann(x_boot, y_boot))
    estimates = np.array(estimates)
    ci_low  = np.percentile(estimates, 100 * alpha / 2)
    ci_high = np.percentile(estimates, 100 * (1 - alpha / 2))
    return ci_low, ci_high

# ==========================================
# 4. Cálculo por cada par significativo
# ==========================================
resultados = []
total      = len(df_sig)

for i, row in df_sig.iterrows():
    centro = row['Centro']
    var    = row['Variable']
    est1   = row['Estacion_1']
    est2   = row['Estacion_2']

    df_centro = df[df['id_centro'] == centro]
    if var in variables_biologicas:
        df_base = df_centro.dropna(subset=variables_biologicas)
    else:
        df_base = df_centro

    x = df_base[df_base['Estacion'] == est1][var].dropna().values
    y = df_base[df_base['Estacion'] == est2][var].dropna().values

    if len(x) > 5 and len(y) > 5:
        # Mediana de cada grupo (contexto)
        med_x = np.median(x)
        med_y = np.median(y)

        # Estimador puntual
        hl    = hodges_lehmann(x, y)

        # IC bootstrap
        ci_low, ci_high = bootstrap_ic(x, y, n_iter=N_BOOTSTRAP)

        # Dirección legible
        if hl > 0:
            direccion = f"{est1} > {est2}"
        elif hl < 0:
            direccion = f"{est1} < {est2}"
        else:
            direccion = "Sin diferencia"

        resultados.append({
            'Centro'              : centro,
            'Variable'            : etiquetas[var],
            'Unidad'              : unidades[var],
            'Estacion_1'          : est1,
            'Estacion_2'          : est2,
            'Mediana_Est1'        : round(med_x, 4),
            'Mediana_Est2'        : round(med_y, 4),
            'Diferencia_HL'       : round(hl, 4),
            'IC95_inferior'       : round(ci_low, 4),
            'IC95_superior'       : round(ci_high, 4),
            'Direccion'           : direccion,
            'Tipo'                : 'Biológica' if var in [etiquetas[v]
                                    for v in variables_biologicas] else 'Oceanográfica'
        })

    progreso = resultados.__len__()
    if progreso % 20 == 0:
        print(f"  Procesados: {progreso} / {total}")

df_hl = pd.DataFrame(resultados)

# ==========================================
# 5. Exportar
# ==========================================
ruta_out = os.path.join(carpeta_stats, "HodgesLehmann_Diferencias_Estacionales.csv")
df_hl.to_csv(ruta_out, index=False)

print(f"\nTabla exportada: {ruta_out}")
print(f"Total de diferencias calculadas: {len(df_hl)}")
print("\nEjemplo de resultados (primeras 10 filas):")
print(df_hl[[
    'Centro', 'Variable', 'Estacion_1', 'Estacion_2',
    'Diferencia_HL', 'IC95_inferior', 'IC95_superior', 'Unidad', 'Direccion'
]].head(10).to_string(index=False))

print("\nCompletado.")

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. Configuración Inicial e Ingesta
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Amplificado_ML.csv"
carpeta_destino = "Resultados_Exploracion/"
os.makedirs(carpeta_destino, exist_ok=True)

df = pd.read_csv(ruta_dataset)

# Variables a comparar
variables = [
    'Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 
    'Velocidad_Corriente_m_s', 'Juveniles', 'Adultos_Moviles', 
    'Hembras_Ovigeras', 'Total_Parasitos_'
]

# Nombres legibles para la consola
nombres_centros = {
    110557: "Rojo (110557)", 
    120069: "Azul (120069)", 
    110745: "Verde (110745)"
}

print("Calculando Diferencias Absolutas (Deltas) entre Centros...\n")

# ==========================================
# 2. Cálculo de Medianas por Centro
# ==========================================
# Calculamos la mediana porque no es afectada por los outliers o asimetrías extremas
df_medianas = df.groupby('id_centro')[variables].median()

# Renombrar el índice para que sea legible
df_medianas.index = df_medianas.index.map(nombres_centros)

print("--- MEDIANAS GLOBALES POR CENTRO ---")
print(df_medianas.round(3).to_string())
print("\n")

# ==========================================
# 3. Cálculo de Diferencias Cruzadas (Deltas)
# ==========================================
# Extraemos las filas como series de pandas
rojo = df_medianas.loc["Rojo (110557)"]
azul = df_medianas.loc["Azul (120069)"]
verde = df_medianas.loc["Verde (110745)"]

# Calculamos las restas
delta_verde_vs_azul = verde - azul
delta_verde_vs_rojo = verde - rojo
delta_rojo_vs_azul = rojo - azul

# Unimos todo en un DataFrame comparativo
df_deltas = pd.DataFrame({
    'Verde vs Azul (Diferencia)': delta_verde_vs_azul,
    'Verde vs Rojo (Diferencia)': delta_verde_vs_rojo,
    'Rojo vs Azul (Diferencia)': delta_rojo_vs_azul
}).round(3)

print("--- DELTAS DE DIFERENCIA ENTRE CENTROS ---")
print("(Nota: Un valor positivo indica que el primer centro tiene más cantidad que el segundo)")
print(df_deltas.to_string())

# ==========================================
# 4. Exportar Resultados
# ==========================================
ruta_exportacion = os.path.join(carpeta_destino, "Comparativa_Numerica_Centros.csv")

# Exportamos unificando ambas tablas
with open(ruta_exportacion, 'w', encoding='utf-8') as f:
    f.write("MEDIANAS GLOBALES POR CENTRO\n")
    df_medianas.round(3).to_csv(f)
    f.write("\nDELTAS DE DIFERENCIA ABSOLUTA\n")
    df_deltas.to_csv(f)

print(f"\nReporte numérico exportado a: {ruta_exportacion}")

In [ ]:
"""
JUSTIFICACIÓN DE SERIES DE TIEMPO — Función de Autocorrelación (ACF)
=====================================================================
Calcula ρ(h) para h = 1 a 12 semanas por variable y centro.
Banda de significancia de Bartlett: ±1.96 / √n

Solo salida por consola, sin gráficos ni archivos.
"""

import pandas as pd
import numpy as np
from scipy import stats
import os

# ==========================================
# 1. Configuración
# ==========================================
ruta_dataset = "data/Dataset_Maestro_Global_ML.csv"

df = pd.read_csv(ruta_dataset)
df['Fecha'] = pd.to_datetime(df['Fecha'])
df = df.sort_values(['id_centro', 'Fecha']).reset_index(drop=True)

df['Velocidad_Corriente_m_s'] = np.sqrt(
    df['Corriente_E_m_s']**2 + df['Corriente_N_m_s']**2
)

centros_objetivo  = [110557, 120069, 110745]
variables_fisicas = ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m',
                     'Altura_Mar_m', 'Velocidad_Corriente_m_s']
variables_bio     = ['Juveniles', 'Adultos_Moviles', 'Hembras_Ovigeras']
todas_variables   = variables_fisicas + variables_bio
MAX_LAG           = 12

# ==========================================
# 2. Función ACF manual
# ==========================================
def calcular_acf(serie, max_lag):
    """
    ρ(h) = γ(h) / γ(0)
    γ(h) = (1/n) Σ (x_t - x̄)(x_{t+h} - x̄)
    Banda Bartlett: ±1.96 / √n
    """
    x   = serie.dropna().values
    n   = len(x)
    mu  = x.mean()
    g0  = np.mean((x - mu) ** 2)         # γ(0) = varianza

    resultados = {}
    for h in range(1, max_lag + 1):
        gh = np.mean((x[:n-h] - mu) * (x[h:] - mu))
        resultados[h] = gh / g0 if g0 > 0 else np.nan

    banda = 1.96 / np.sqrt(n)
    return resultados, banda, n

# ==========================================
# 3. Cálculo y salida por consola
# ==========================================
print("=" * 70)
print("FUNCIÓN DE AUTOCORRELACIÓN (ACF) — Justificación de Series de Tiempo")
print("Fórmula: ρ(h) = γ(h)/γ(0)   |   Sig: ±1.96/√n")
print("=" * 70)

resumen = []   # Para el resumen final

for centro in centros_objetivo:
    df_centro = df[df['id_centro'] == centro]
    df_bio    = df_centro.dropna(subset=variables_bio)

    print(f"\n{'─'*70}")
    print(f"  CENTRO {centro}")
    print(f"{'─'*70}")

    for var in todas_variables:
        df_base = df_bio if var in variables_bio else df_centro
        serie   = df_base[var].dropna()
        n       = len(serie)

        if n < 20:
            print(f"  {var:<30s} → n insuficiente ({n})")
            continue

        acf, banda, n = calcular_acf(serie, MAX_LAG)
        n_sig = sum(1 for h, rho in acf.items() if abs(rho) > banda)

        # Línea compacta con todos los lags
        lags_str = "  ".join([
            f"h{h}={'*' if abs(rho) > banda else ' '}{rho:+.3f}"
            for h, rho in acf.items()
        ])

        print(f"\n  {var}")
        print(f"  n={n}  |  banda=±{banda:.4f}  |  lags sig: {n_sig}/{MAX_LAG}")
        print(f"  {lags_str}")

        resumen.append({
            'Centro'    : centro,
            'Variable'  : var,
            'n'         : n,
            'Banda'     : round(banda, 4),
            'Lags_sig'  : n_sig,
            'rho_h1'    : round(acf[1], 4),
            'rho_h4'    : round(acf[4], 4),
            'rho_h12'   : round(acf[12], 4),
            'Conclusion': 'Memoria temporal' if n_sig >= 3 else 'Débil estructura'
        })

# ==========================================
# 4. Resumen final
# ==========================================
print(f"\n{'=' * 70}")
print("RESUMEN — Variables con memoria temporal significativa (≥3 lags sig.)")
print(f"{'=' * 70}")
print(f"{'Centro':<10} {'Variable':<30} {'h=1':>7} {'h=4':>7} {'h=12':>7} {'Sig/12':>7}  Conclusión")
print(f"{'─'*80}")

for r in resumen:
    print(
        f"  {r['Centro']:<8} {r['Variable']:<30} "
        f"{r['rho_h1']:>+7.3f} {r['rho_h4']:>+7.3f} {r['rho_h12']:>+7.3f} "
        f"  {r['Lags_sig']:>2}/{12}    {r['Conclusion']}"
    )

total     = len(resumen)
con_mem   = sum(1 for r in resumen if r['Conclusion'] == 'Memoria temporal')
print(f"\nVariables con memoria temporal: {con_mem} de {total}")
print("\nJustificación: si ρ(h) > ±1.96/√n para múltiples lags,")
print("los datos tienen dependencia temporal y el análisis de series")
print("de tiempo es el enfoque estadístico correcto (Shumway & Stoffer, 2011).")